# TODO:

Politics / Mentions: two promising wallets
0x9e8ecc4cb3c4e48f544cba2fbbb252a6a65e8db8	
0x0cb10c40b0776e9ee8cef970af85724654dda76c


# Wallet Strategy Selection

This stage tracks two distinct wallet groups on the full trade stream (BUY + SELL):

- `copyable_group`: wallets that pass base eligibility and strong copyability thresholds.
- `predicting_group`: the remaining eligible wallets (not in copyable) that pass a predictability score threshold.

Notes:
- Group membership is threshold-based (absolute scores), not target-count based.
- Open-buy skill metrics are kept for diagnostics/legacy comparison.
- `selected_wallets_v2.parquet` stores both groups with a `wallet_group` label.
- Stage2 backtest uses only `predicting_group`.
- A post-selection **size-share overlay** (`copyable_pnl_size`) keeps BUY fills whose notional is small relative to the prior-60s market volume; see the *Size-share filter* section.
- Candidate universe uses the original 2-day lead window; the flow rule later uses the 0-7 day window.
- An all-wallet `copyable_pnl_price_size` variant ports the chosen `stage1_flow_selection` filter exactly (fit-q50 size-share threshold, fit/val wallet screen) for both cohorts; see the *Flow-selection filter* section.


In [1]:
%load_ext autoreload
%autoreload 2

import datetime
from pathlib import Path
import json

import numpy as np
import pandas as pd
import pyarrow.dataset as ds

np.set_printoptions(precision=4, suppress=True)
pd.options.display.float_format = '{:.2f}'.format

## Configuration

In [2]:
from enum import Enum

class MarketFilterMode(Enum):
    DEFAULT = "default"
    SPORTS = "sports"
    CRYPTO = "crypto"

# ── MARKET FILTER MODE SWITCH ────────────────────────────────────────────────
MARKET_FILTER_MODE = MarketFilterMode.SPORTS

# ── configuration ─────────────────────────────────────────────────────────────
# TAGS = set(['Weather'])
# TAGS = set(['Esports'])
TAGS = set(['Sports'])
# TAGS = set(['Awards'])
# TAGS = set(['MrBeast', 'Trump', 'Elections', 'Celebrities', 'Elon Musk', 'YouTube'])

RECENCY_DAYS = 90

MAX_LEAD_DAYS = 7
if 'Politics' in TAGS:
    MAX_LEAD_DAYS = 14

PRICE_BUCKET_BINS   = [0.0, 0.1, 0.25, 0.4, 0.6, 0.75, 0.9, 1.0]
PRICE_BUCKET_LABELS = [
    "0.00-0.10", "0.10-0.25", "0.25-0.40", "0.40-0.60",
    "0.60-0.75", "0.75-0.90", "0.90-1.00",
]

if MARKET_FILTER_MODE == MarketFilterMode.SPORTS:
    TRADES_DIR    = Path("../../data/polygon_trades_processed_sports")
    WORKSPACE_DIR = Path("../../data/trade_signals_workspace_v2_sports")
elif MARKET_FILTER_MODE == MarketFilterMode.CRYPTO:
    TRADES_DIR    = Path("../../data/polygon_trades_processed_crypto")
    WORKSPACE_DIR = Path("../../data/trade_signals_workspace_v2_crypto")
else:
    TRADES_DIR    = Path("../../data/polygon_trades_processed")
    WORKSPACE_DIR = Path("../../data/trade_signals_workspace_v2")
WORKSPACE_DIR.mkdir(parents=True, exist_ok=True)

dataset = ds.dataset(TRADES_DIR, format="parquet")


WALLET_FILTER = None
# WALLET_FILTER = [
#     '0x919698b19427cbe6945b0dc823f2d9e126a4d934',
#     '0x85d22a973e817f87afe0a7fd346a39823cce7479',
#     '0x6011655c4afb76f36dd1b08a137a1ba73466b31e',
#     '0x5f612351a9e46afbe3f164473a7d77a69ec9840a',
# ]
# ── exposure thresholds (bankroll-discipline carve, signal_lab §14-18) ─────
# Copy only trades that happen in high exposure (position * price) is >= T_{w,f},
# (fit via wallet_capture_thresholds on train; f == 0 copies everything).
EXPOSURE_FRACS   = [1] #, 0.95, 0.9, 0.8, 0.5, 0.2, 0.1, 0.05)
EXPOSURE_PNL_COL = "copyable_pnl"
EXPOSURE_EXP_COL = "exposure"   # computed as position * price

EXPOSURE_FILTER_FRAC = 0.2

DATASET_COLUMNS        = set(dataset.schema.names)
TRIGGER_TX_HASH_COLUMN = (
    "transaction_hash" if "transaction_hash" in DATASET_COLUMNS
    else ("tx_hash" if "tx_hash" in DATASET_COLUMNS else None)
)


# ── raw trades directory (for flow feature computation) ─────────────────
if MARKET_FILTER_MODE == MarketFilterMode.SPORTS:
    RAW_TRADES_DIR = Path("../../data/trades_polygon_sports")
elif MARKET_FILTER_MODE == MarketFilterMode.CRYPTO:
    RAW_TRADES_DIR = Path("../../data/trades_polygon_crypto")
else:
    RAW_TRADES_DIR = Path("../../data/trades_polygon")

# ── flow signal filter (post-selection overlay) ──────────────────────────
REBUILD_FLOW = False
FLOW_PRICE_LO = 0.20
FLOW_PRICE_HI = 0.50
FLOW_SHARE_THR = -0.10
FLOW_STRICT = False
FLOW_CARVE_FRAC = EXPOSURE_FILTER_FRAC

# ── size-share filter (post-selection overlay) ──────────────────────────
# Ex-ante: trade notional / prior-60s total market notional. Small values
# mean the wallet's BUY is small relative to recent volume, so a copy is
# less likely to move the price and more likely to fill well.
SIZE_SHARE_PRICE_LO = 0.30
SIZE_SHARE_PRICE_HI = 0.75
SIZE_SHARE_QUANTILE = 0.50

# ── flow-selection filter (port of stage1_flow_selection) ─────────────────
# Exact port of the chosen ex-ante market situation from stage1_flow_selection.
# Self-contained constants — deliberately NOT shared with the SIZE_SHARE_*
# predicting-group overlay above, so tuning one never silently changes the
# other.
FLOWSEL_FIT_END = pd.Timestamp("2026-08-01", tz="UTC")
FLOWSEL_VAL_END = pd.Timestamp("2026-09-01", tz="UTC")
FLOWSEL_PRICE_BAND = (0.30, 0.75)
FLOWSEL_SIZE_SHARE_QUANTILE = 0.50
FLOWSEL_MAX_LEAD_DAYS = 7
FLOWSEL_MIN_LEAD_DAYS = 0
# Candidate-universe lead window (the original strategy-selection 2-day
# window); the flow-selection rule itself keeps the 0-7 day window.
FLOWSEL_CANDIDATE_MAX_LEAD_DAYS = 2
FLOWSEL_MIN_FIT_WALLET_FILLS = 20
FLOWSEL_MIN_VAL_WALLET_FILLS = 10


## Derive train/test dates from data

## Full-stream wallet metrics for copy-trigger selection

Build per-wallet metrics on the full training trade stream (BUY + SELL).
Copyability is considered here (stage1), not in stage0 filtering.


In [3]:
from polymarket_analysis.wallet_selection.volatility import (
    compute_wallet_metrics,
    filter_wallets_by_volatility,
)

from polymarket_analysis.data.data_catalogue import load_markets_processed
mdf = load_markets_processed()

print(mdf.groupby('primary_tag').size().sort_values(ascending=False).head(20))

_tagset = set(TAGS)
mdf = mdf[
    (True if TAGS is None else mdf['tags'].apply(lambda tags: bool(_tagset & set(tags))))
    & ~(mdf['primary_tag'].isin(['Crypto']))
    & ((TAGS is None) or mdf["tags"].apply(lambda tags: any(tag in TAGS for tag in tags)))
    & (mdf['winner_token_id'].notna())
].copy().reset_index(drop=True)

print(f'selected markets for tags {TAGS}: {len(mdf)}')

primary_tag
Sports         3678528
Crypto         1082612
Weather         191014
Politics        103129
Finance          58994
Awards           12364
Movies           11521
AI                8498
Music             7778
Culture           6947
All               3808
Esports           3699
Business          3452
MrBeast           2302
Science           1868
Iran              1788
Celebrities       1447
Trump             1259
Tech              1018
Elections          928
dtype: int64
selected markets for tags {'Sports'}: 3265838


In [4]:
# Load full processed trades for the full-stream path
trade_files = sorted(TRADES_DIR.glob("*.parquet"))
df_full = pd.concat(
    [pd.read_parquet(f).merge(mdf, on="condition_id", how="inner") for f in trade_files], ignore_index=True
    )

print(f"Full loaded df_full size: {len(df_full)}")

# df_full = df_full.merge(mdf, on="condition_id", how="inner")

df_full['outcome'] = df_full['outcome_x']
del df_full['outcome_x'], df_full['outcome_y']

df_full["dt"] = pd.to_datetime(df_full["dt"], utc=True)

print(f"rows after mdf merge (BUY + SELL): {len(df_full):,}")

lead = (
    pd.to_datetime(df_full["last_condition_trade_ts"], utc=True, errors="coerce")
    - df_full["dt"]
)
df_full = df_full[lead <= pd.Timedelta(days=MAX_LEAD_DAYS)].copy()
print(
    f"Lead filter (<= {MAX_LEAD_DAYS}d before resolution): "
    f"{len(df_full):,} trades"
)

# Normalise grouped schema → canonical fill-level names
if "total_quantity" in df_full.columns and "quantity" not in df_full.columns:
    df_full = df_full.rename(columns={
        "total_quantity": "quantity",
        "avg_price": "price",
        "trade_value_usdc": "usdc_amount",
    })

df_full["usdc_amount"] = df_full["usdc_amount"].astype(float)
df_full["final_value_usdc"] = df_full["final_value_usdc"].astype(float)
df_full["quantity"] = df_full["quantity"].astype(float)

# Full-stream PnL / notional across BUY + SELL
df_full["pnl"] = np.where(
    df_full["side"] == "BUY",
    df_full["final_value_usdc"] - df_full["usdc_amount"],
    df_full["usdc_amount"] - df_full["final_value_usdc"],
)
df_full["notional"] = np.where(
    df_full["side"] == "BUY",
    df_full["usdc_amount"],
    df_full["quantity"] * (1 - df_full["price"].astype(float)),
)

# Ex-ante size feature: current BUY notional / prior-60s total notional.
df_full["size_share"] = (
    df_full["usdc_amount"].astype(float)
    / df_full["usdc_all_60"].astype(float).replace(0, np.nan)
)
TRAIN_END = pd.Timestamp("2026-09-01", tz="UTC")
df_full['is_train'] = df_full['last_condition_trade_ts'] <= TRAIN_END

_n_wallets = df_full['wallet'].nunique()
_train = df_full[df_full['is_train']]
_test_wallets = set(df_full.loc[~df_full['is_train'], 'wallet'].unique())
_train_stats = _train.groupby('wallet').agg(
    train_pnl=('pnl', 'sum'),
    train_markets=('condition_id', 'nunique'),
)
del _train
_eligible = _train_stats.index[
    _train_stats.index.isin(_test_wallets)
    & (_train_stats['train_pnl'] >= 1000)
    & (_train_stats['train_markets'] >= 20)
]
df_full = df_full[df_full['wallet'].isin(_eligible)].copy()
print(f"Basic wallet filter: {df_full['wallet'].nunique():,} of {_n_wallets:,} wallets kept")

# FULL (uncarved) wallet metrics — computed before the exposure carve so
# both stat sets exist: full stats are passed to the selection filters,
# carved stats keep driving the execution/diagnostics stream below.
df_all_train_full = df_full[df_full["is_train"]].copy()
wallet_vol_all_full, _ = compute_wallet_metrics(df_all_train_full)
del df_all_train_full

df_buy_train_full = df_full[df_full["is_train"] & (df_full["side"] == "BUY")].copy()
wallet_vol_train_full, _ = compute_wallet_metrics(df_buy_train_full)
del df_buy_train_full

wallet_vol_train_full = wallet_vol_train_full.merge(
    wallet_vol_all_full[["wallet", "max_drawdown", "total_pnl"]].rename(
        columns={"max_drawdown": "max_drawdown_all", "total_pnl": "total_pnl_all"}
    ),
    on="wallet",
    how="left",
)
wallet_vol_train_full["max_drawdown_all_to_pnl_all"] = np.where(
    wallet_vol_train_full["total_pnl_all"] > 0,
    wallet_vol_train_full["max_drawdown_all"] / wallet_vol_train_full["total_pnl_all"],
    float("nan"),
)

# Lean pre-carve BUY stream handed to the explorer: same rows as `df_full`
# before the exposure carve, flagged by `is_carved`, so the PnL panels can
# overlay full and carved variants.
_full_stream_cols = [
    "wallet", "condition_id", "token_id", "tx_hash", "dt", "side",
    "last_condition_trade_ts", "position", "price", "notional", "pnl",
    "copyable_pnl", "is_train", "copyable_qty_5m_100", "is_carved",
    "flow_own_usdc_60", "usdc_all_60", "size_share", "price_chg_last_own",
]

if EXPOSURE_FILTER_FRAC is not None:
    from polymarket_analysis.wallet_selection.exposure import wallet_capture_thresholds

    df_full["deployed_capital"] = df_full["position"] * df_full["price"]

    _fit = df_full[df_full["is_train"] & (df_full["side"] == "BUY")]
    _capture = wallet_capture_thresholds(
        _fit, (EXPOSURE_FILTER_FRAC,), EXPOSURE_PNL_COL, "deployed_capital"
    )
    del _fit
    _T = df_full["wallet"].map(_capture.set_index("wallet")["threshold"]).fillna(0.0)

    _keep = df_full["deployed_capital"].notna() & (df_full["deployed_capital"] >= _T)
    df_full["is_carved"] = _keep

    df_full_buy_full = df_full.loc[
        df_full["side"] == "BUY", _full_stream_cols
    ].copy()

    _n = len(df_full)
    df_full = df_full[_keep].copy()
    print(
        f"Exposure carve (frac {EXPOSURE_FILTER_FRAC:g}): "
        f"{len(df_full):,} of {_n:,} trades kept"
    )
else:
    df_full["is_carved"] = True
    df_full_buy_full = df_full.loc[
        df_full["side"] == "BUY", _full_stream_cols
    ].copy()

# TRAIN_START = TRAIN_END - pd.Timedelta(days=30)
# df_full = df_full[df_full['dt'] >= TRAIN_START]

# Carved all-trades (BUY + SELL) wallet metrics: drawdown incl. SELL legs
df_all_train = df_full[df_full["is_train"]].copy()
wallet_vol_all, _ = compute_wallet_metrics(df_all_train)
del df_all_train

# Primary stream stays BUY-only (carved execution/diagnostics stream)
df_full = df_full[df_full["side"] == "BUY"].copy().reset_index(drop=True)

df_slice = df_full[df_full["is_train"]].copy()

wallet_vol_train, _ = compute_wallet_metrics(df_slice)
del df_slice

wallet_vol_train = wallet_vol_train.merge(
    wallet_vol_all[["wallet", "max_drawdown", "total_pnl"]].rename(
        columns={"max_drawdown": "max_drawdown_all", "total_pnl": "total_pnl_all"}
    ),
    on="wallet",
    how="left",
)
wallet_vol_train["max_drawdown_all_to_pnl_all"] = np.where(
    wallet_vol_train["total_pnl_all"] > 0,
    wallet_vol_train["max_drawdown_all"] / wallet_vol_train["total_pnl_all"],
    float("nan"),
)

# Flow 
df_full["flow_usdc_share_60"] = (
    df_full["flow_own_usdc_60"]
    / df_full["usdc_all_60"].replace(0, np.nan)
)

print(len(wallet_vol_train))
print(len(wallet_vol_train_full))


Full loaded df_full size: 24740693
rows after mdf merge (BUY + SELL): 24,740,693
Lead filter (<= 7d before resolution): 24,535,398 trades
Basic wallet filter: 2,694 of 21,251 wallets kept


/Users/vobornij/projects/polymarket/src/polymarket_analysis/wallet_selection/volatility.py:190: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  piv = df.pivot_table(
/Users/vobornij/projects/polymarket/src/polymarket_analysis/wallet_selection/volatility.py:190: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  piv = df.pivot_table(


Exposure carve (frac 0.2): 3,247,201 of 13,407,748 trades kept


/Users/vobornij/projects/polymarket/src/polymarket_analysis/wallet_selection/volatility.py:190: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  piv = df.pivot_table(
/Users/vobornij/projects/polymarket/src/polymarket_analysis/wallet_selection/volatility.py:190: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  piv = df.pivot_table(


2694
2694


In [5]:
daily_stats = (
    df_full.groupby(pd.Grouper(key="dt", freq="D"))
    .agg(
        trade_count=("price", "size"),
        total_volume=("price", lambda s: (s * df_full.loc[s.index, "quantity"]).sum()),
    )
    .reset_index()
)

# import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.graph_objects as go


# Create plot with secondary y-axis
fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Scatter(
        x=daily_stats["dt"],
        y=daily_stats["trade_count"],
        name="Number of trades",
        mode="lines+markers",
    ),
    secondary_y=False,
)

fig.add_trace(
    go.Scatter(
        x=daily_stats["dt"],
        y=daily_stats["total_volume"],
        name="Total volume (price × qty)",
        mode="lines+markers",
    ),
    secondary_y=True,
)

fig.update_layout(
    title="Daily Trade Count and Total Volume",
    xaxis_title="Date",
    template="plotly_white",
    hovermode="x unified",
)

fig.update_yaxes(title_text="Number of trades", secondary_y=False)
fig.update_yaxes(title_text="Total volume", secondary_y=True)

fig.show()


In [6]:
len(df_full[df_full["is_train"]])

1674556

In [7]:
# mdf = load_markets_processed()
# df_full = pd.concat(
#     [
#         (
#             pd.read_parquet(f)
#             .merge(mdf, on="condition_id", how="inner")
#             .assign(dt=lambda df: pd.to_datetime(df["dt"], utc=True))
#             .loc[lambda df: df["dt"] >=  datetime.datetime(2026, 7, 1, tzinfo=datetime.timezone.utc)]
#         )
#         for f in trade_files
#     ],
#     ignore_index=True,
# )


In [8]:
print (df_full.columns)
print(df_full.index.names)

Index(['wallet', 'condition_id', 'token_id', 'dt', 'side', 'position',
       'quantity', 'price', 'usdc_amount', 'final_value_usdc', 'trade_pnl',
       'copyable_pnl', 'token_winner', 'final_price',
       'last_condition_trade_ts', 'tx_hash', 'num_fills', 'is_train',
       'copyable_qty_5m_100', 'avail_copy_qty_5m_100',
       'avail_copy_total_vol_5m_100', 'avail_copy_count_5m_100',
       'flow_own_usdc_60', 'usdc_all_60', 'price_chg_last_own', 'end_date_iso',
       'question', 'tags', 'primary_tag', 'winner_token_id', 'outcome', 'pnl',
       'notional', 'size_share', 'deployed_capital', 'is_carved',
       'flow_usdc_share_60'],
      dtype='object')
[None]


In [9]:
_copy_scale = np.minimum(2, df_full['avail_copy_qty_5m_100'] / df_full['quantity'])
df_full['copyable_qty_2x'] = df_full['copyable_qty_5m_100'] * _copy_scale
df_full['copyable_pnl_2x'] = df_full['copyable_pnl'] * _copy_scale

_copy_scale = np.minimum(0.5, df_full['avail_copy_qty_5m_100'] / df_full['quantity'])

df_full['copyable_qty_0.5x'] = df_full['copyable_qty_5m_100'] * _copy_scale
df_full['copyable_pnl_0.5x'] = df_full['copyable_pnl'] * _copy_scale

In [10]:
# ── Copyable PnL variants ─────────────────────────────────────────────────────
# Per-fill copyable PnL under different lookback-window / fill-price assumptions.
# stage0 exports per-fill qty columns only for the variants enabled in its
# VARIANTS config (currently just the 5m primary); keep only the variants
# whose qty column is present in the current dataset.
COPYABLE_PNL_BASE = "copyable_pnl"
COPYABLE_VARIANTS = {
    # "5m_095": dict(qty_col="copyable_qty_5m_095", fill_factor=0.95),
    # "20m_100": dict(qty_col="copyable_qty_20m_100", fill_factor=1.0),
    # "2x": dict(qty_col="copyable_qty_2x", fill_factor=1.0),
    # "0.5x": dict(qty_col="copyable_qty_0.5x", fill_factor=1.0),
}
COPYABLE_VARIANTS = {
    k: cfg for k, cfg in COPYABLE_VARIANTS.items() if cfg["qty_col"] in df_full.columns
}
print(f"copyable PnL variants in use: {list(COPYABLE_VARIANTS)}")

for key, cfg in COPYABLE_VARIANTS.items():
    qty = df_full[cfg["qty_col"]]
    df_full[f"copyable_pnl_{key}"] = np.where(
        df_full["side"] == "BUY",
        (df_full["final_price"] - df_full["price"] * cfg["fill_factor"]) * qty,
        (df_full["price"] / cfg["fill_factor"] - df_full["final_price"]) * qty,
    )

COPYABLE_PNL_COLS = [COPYABLE_PNL_BASE] + [f"copyable_pnl_{k}" for k in COPYABLE_VARIANTS]
COPYABLE_PNL_VARIANTS = COPYABLE_PNL_COLS[1:]

copyable PnL variants in use: []


In [11]:
df_full['copyable_notional'] = df_full['notional'] * (df_full['copyable_pnl'] / df_full['pnl'].replace(0, np.nan))
df_full['roi'] = df_full['pnl'] / df_full['notional'].replace(0, np.nan)
df_full['copyable_roi'] = df_full['copyable_pnl'] / df_full['copyable_notional'].replace(0, np.nan)

df_full.head()

,wallet,condition_id,token_id,dt,side,position,quantity,price,usdc_amount,final_value_usdc,...,deployed_capital,is_carved,flow_usdc_share_60,copyable_qty_2x,copyable_pnl_2x,copyable_qty_0.5x,copyable_pnl_0.5x,copyable_notional,roi,copyable_roi
0,0x4fc19a907012ef58717ee7ceb1f89057f71aec92,0x0f11f83fb8e412686efcb9430fcbeb326e91f24f4f90...,4095939065153142593574376695211848274631622064...,2025-01-01 00:33:09+00:00,BUY,229.15,229.15,0.65,148.95,229.15,...,148.95,True,0.32,0.00,0.00,0.00,0.00,0.00,0.54,0.54
1,0x4fc19a907012ef58717ee7ceb1f89057f71aec92,0x0f11f83fb8e412686efcb9430fcbeb326e91f24f4f90...,4095939065153142593574376695211848274631622064...,2025-01-01 00:46:17+00:00,BUY,999.98,770.83,0.65,501.04,770.83,...,649.99,True,1.93,0.25,0.09,0.25,0.09,9.03,0.54,0.54
2,0x8f19df5088b81721175218d225443ec28e55b0ca,0x0f11f83fb8e412686efcb9430fcbeb326e91f24f4f90...,2760237849528716269099862867806998297850735825...,2025-01-01 04:13:25+00:00,BUY,77.08,77.08,0.01,0.77,0.00,...,0.77,True,-0.17,0.00,-0.00,0.00,-0.00,0.00,-1.00,NaN
3,0x8f19df5088b81721175218d225443ec28e55b0ca,0x0f11f83fb8e412686efcb9430fcbeb326e91f24f4f90...,2760237849528716269099862867806998297850735825...,2025-01-01 04:13:27+00:00,BUY,85.49,8.41,0.01,0.08,0.00,...,0.85,True,-0.17,0.00,-0.00,0.00,-0.00,0.00,-1.00,NaN
4,0x8f19df5088b81721175218d225443ec28e55b0ca,0x0f11f83fb8e412686efcb9430fcbeb326e91f24f4f90...,2760237849528716269099862867806998297850735825...,2025-01-01 04:13:33+00:00,BUY,999.99,914.50,0.01,9.14,0.00,...,10.00,True,-0.17,0.00,-0.00,0.00,-0.00,0.00,-1.00,NaN


In [12]:
wallet_vol_test, _ = compute_wallet_metrics(df_full[~df_full["is_train"]].copy())

/Users/vobornij/projects/polymarket/src/polymarket_analysis/wallet_selection/volatility.py:190: FutureWarning:

The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior



In [13]:
df_full['bucket'] = df_full['dt'].dt.round("5min")

In [14]:
for _wv in (wallet_vol_train, wallet_vol_train_full):
    _wv['copyable_pnl_factor'] = np.clip(
        _wv['copyable_pnl'] / _wv['total_pnl'].replace(0, np.nan),
        0,
        1.0,
    ).fillna(0.0)
    _wv['copyable_roi'] = _wv['average_roi'] * _wv['copyable_pnl_factor']
    _wv['buy_copyable_roi'] = _wv['buy_copyable_pnl'] / _wv['buy_copyable_notional'].replace(0, np.nan)

# Keep sorting deterministic for downstream filtering/inspection.
wallet_vol_train.sort_values('copyable_roi', ascending=False, inplace=True)
wallet_vol_train.reset_index(drop=True, inplace=True)
wallet_vol_train_full.sort_values('copyable_roi', ascending=False, inplace=True)
wallet_vol_train_full.reset_index(drop=True, inplace=True)


In [15]:
wallet_cohorts = {}

In [16]:
# Final wallet selection: threshold-based copyable + predicting groups
# COPYABLE_SCORE_MIN = -1
# PREDICTING_SCORE_MIN = -2

candidates = wallet_vol_train.copy()
candidates['buy_copyable_factor'] = candidates['buy_copyable_pnl'] / candidates['buy_pnl']

# Additional filter: keep only wallets active in the test period (> 0 test trades)
test_active_wallets = set(df_full.loc[~df_full['is_train'], 'wallet'].unique())
n_candidates = len(candidates)
candidates = candidates[candidates['wallet'].isin(test_active_wallets)].copy()
print(f"Test-active filter: kept {len(candidates):,} of {n_candidates:,} candidates")

for col, default in {
    'top10_pnl_pct': np.nan,
    'top_market_abs_pnl_pct': np.nan,
    'market_pnl_hhi': np.nan,
    'positive_bucket_share': np.nan,
}.items():
    if col not in candidates.columns:
        candidates[col] = default

for col in [
    'average_roi', 'median_roi', 'num_buckets', 'num_markets',
    'pnl_volatility', 'max_drawdown_to_pnl',
    'top_market_pnl_pct', 'top_market_abs_pnl_pct', 'top5_pnl_pct', 'top10_pnl_pct', 'market_pnl_hhi',
    'copyable_roi', 'copyable_pnl_factor', 'copyable_pnl', 'total_notional',
    'max_copyable_drawdown_to_copyable_pnl', 'worst5_pnl_pct', 'positive_bucket_share',
]:
    if col in candidates.columns:
        candidates[col] = pd.to_numeric(candidates[col], errors='coerce')

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "wallet_selection"))

from signal_lab.filters import (
    select_strategy_eligible,
    select_strategy_selection,
)

if "Esports" in TAGS or "Sports" in TAGS:
    variant = "sports"
elif "Politics" in TAGS:
    variant = "politics"
else:
    variant = "weather"

# Two separate stat frames passed to the filter functions: the full
# (non-filtered) stats drive the mask rules for now; the carved stats are
# available as the second parameter for rule-level adjustments.
eligible_wallets = select_strategy_eligible(
    wallet_vol_train_full, wallet_vol_train, variant=variant
)
copyable_wallets = select_strategy_selection(
    wallet_vol_train_full, wallet_vol_train, variant=variant
)

eligible_base = candidates[candidates['wallet'].isin(eligible_wallets)].copy()
if eligible_base.empty:
    raise ValueError('No wallets passed base eligibility filters.')

copyable_candidates = eligible_base[eligible_base['wallet'].isin(copyable_wallets)].copy()
copyable_candidates['copyable_efficiency'] = copyable_candidates['copyable_pnl'].fillna(0.0) / (copyable_candidates['total_notional'].fillna(0.0) + 1.0)
copyable_candidates['copyable_dd_ratio'] = (
    copyable_candidates['max_copyable_drawdown_to_copyable_pnl']
    .fillna(copyable_candidates['max_drawdown_to_pnl'])
    .fillna(0.0)
)


wallet_cohorts['copyable_group'] = (
    copyable_candidates
    .reset_index(drop=True)
)

predicting_pool = eligible_base[
    ~eligible_base['wallet'].isin(wallet_cohorts['copyable_group']['wallet'])
].copy()
wallet_cohorts['predicting_group'] = (
    predicting_pool
    .reset_index(drop=True)
)

overlap = set(wallet_cohorts['copyable_group']['wallet']) & set(wallet_cohorts['predicting_group']['wallet'])
if overlap:
    raise ValueError(f'Wallet groups must be disjoint; found overlap of {len(overlap)} wallets.')

wallet_cohorts['multi_filter'] = wallet_cohorts['copyable_group'].copy()

# Split the copyable group into 3 terciles by estimated_copyable_buy_sharpe.
# qcut raises when the values cannot form 3 unique bins (e.g. all-NaN sharpe
# after heavy filtering); skip the split and keep empty tercile cohorts so
# the summary print below still works.
_sharpe = wallet_cohorts['copyable_group']['estimated_copyable_buy_sharpe']
_SHARPE_LABELS = ['copyable_buy_sharpe_low', 'copyable_buy_sharpe_mid', 'copyable_buy_sharpe_high']
try:
    _terciles = pd.qcut(_sharpe, 3, labels=_SHARPE_LABELS)
except ValueError as _err:
    _terciles = None
    print(f"Sharpe terciles skipped: {_err}")
    print(
        f"  estimated_copyable_buy_sharpe: {_sharpe.notna().sum()}/{len(_sharpe)} valid, "
        f"{_sharpe.dropna().nunique()} distinct values"
    )
for _label in _SHARPE_LABELS:
    wallet_cohorts[_label] = (
        wallet_cohorts['copyable_group'][_terciles == _label].copy().reset_index(drop=True)
        if _terciles is not None
        else wallet_cohorts['copyable_group'].iloc[0:0].copy()
    )


print(f"Base-eligible wallets: {len(eligible_base):,}")
print(f"Copyable candidates: {len(copyable_candidates):,}")
print(f"Selected wallets (copyable group): {len(wallet_cohorts['copyable_group']):,} total copyable pnl {wallet_cohorts['copyable_group']['copyable_pnl'].sum()}")
print(f"Selected wallets (predicting group): {len(wallet_cohorts['predicting_group']):,} total predicting group copyable pnl {wallet_cohorts['predicting_group']['copyable_pnl'].sum()}")
print("Copyable sharpe terciles: " + ", ".join(f"{k}={len(wallet_cohorts[k])}" for k in ['copyable_buy_sharpe_low', 'copyable_buy_sharpe_mid', 'copyable_buy_sharpe_high']))
wallet_cohorts['copyable_group'].sort_values('estimated_copyable_buy_sharpe', key=abs, ascending=False).head(10)



Test-active filter: kept 1,114 of 2,694 candidates
Base-eligible wallets: 38
Copyable candidates: 26
Selected wallets (copyable group): 26 total copyable pnl 220158.25385144874
Selected wallets (predicting group): 12 total predicting group copyable pnl 64004.987667106056
Copyable sharpe terciles: copyable_buy_sharpe_low=8, copyable_buy_sharpe_mid=8, copyable_buy_sharpe_high=8


,wallet,num_buckets,trade_count,total_notional,total_pnl,copyable_pnl,num_markets,median_dt,median_roi,average_roi,...,return,max_drawdown_all,total_pnl_all,max_drawdown_all_to_pnl_all,copyable_pnl_factor,copyable_roi,buy_copyable_roi,buy_copyable_factor,copyable_efficiency,copyable_dd_ratio
14,0x46ca395d519ecc8fa9a4fa5c77917f762cf8ac4c,14,74,22290.53,4950.79,2345.18,11,2026-08-23 07:10:00+00:00,0.17,0.31,...,0.22,736.77,2628.33,0.28,0.47,0.15,0.26,0.47,0.11,0.00
11,0x3d0a2cfe37769d915b40c49fe85b3602d53e1bd3,5,5,7278.35,1084.33,837.85,4,2026-08-10 14:10:00+00:00,0.18,0.26,...,0.15,1629.69,-545.36,NaN,0.77,0.20,0.14,0.77,0.12,0.00
16,0xaa00bd99d8c8c3f0221c4d10803f7d6c95a3f736,2,2,20797.02,5874.38,2866.25,2,2026-05-13 13:45:00+00:00,0.28,0.28,...,0.28,13.67,5860.71,0.00,0.49,0.14,0.27,0.49,0.14,0.00
8,0x55c137e7bb089e172b41e5420d24f303ce696485,7,7,5829.21,3956.70,1657.00,5,2026-06-27 18:40:00+00:00,0.54,0.67,...,0.68,214.99,3741.70,0.06,0.42,0.28,0.69,0.42,0.28,0.00
7,0xf3ce7f04dde4f8c5aed19a20e5ecd8520e5ca57a,15,16,179025.99,62010.25,51200.92,15,2026-07-11 14:05:00+00:00,0.30,0.36,...,0.35,0.00,62010.25,0.00,0.83,0.29,0.35,0.83,0.29,0.00
1,0x1b9ccfe8ee4597f6a7d8ed86761390cc43a2b67d,4,9,5546.33,5616.94,4103.01,3,2026-08-19 23:15:00+00:00,1.03,0.97,...,1.01,0.00,5616.94,0.00,0.73,0.71,1.03,0.73,0.74,0.00
9,0x22761026b64f1f31e9d0ca7318c86922c454efd0,22,52,19700.15,2110.35,1361.02,9,2026-07-05 03:20:00+00:00,0.43,0.41,...,0.11,5061.59,2110.35,2.40,0.64,0.26,0.58,0.64,0.07,0.00
6,0x24ae647bd3d6fec6ec54e3e02606a0a260804c80,7,19,43417.54,9718.61,8301.57,7,2026-07-06 21:00:00+00:00,0.06,0.44,...,0.22,1837.15,7041.68,0.26,0.85,0.38,0.32,0.85,0.19,0.00
19,0x2edd21765f4ec11beb6e36b02324a6f6f33ce968,4,4,5580.00,1540.38,520.47,3,2026-08-09 00:52:30+00:00,0.30,0.28,...,0.28,0.00,1540.38,0.00,0.34,0.10,0.29,0.34,0.09,0.00
22,0xb8d757413fe4d4ec4716390938f12100614f5ac7,39,50,31732.85,14538.56,2309.84,6,2026-07-31 15:35:00+00:00,0.41,0.46,...,0.46,4624.99,9913.57,0.47,0.16,0.07,0.46,0.16,0.07,0.00


Fine-tune the selection in the browser: every numeric stat of both stat frames
(`wallet_vol_train_full` → `__full`, `wallet_vol_train` → `__carved`) gets a
dual range slider, pre-set to the `variant` thresholds. Click a wallet for its
train/test PnL panels, or switch to **picked aggregate** to see the cumulative
PnL of all lassoed wallets summed together.

The PnL panels use **checkable fill filters** (price band, size share, flow
share, carved-only, copyable-only). Every checked filter is combined into a
single per-fill mask and the wallet / copyable cumulative PnL is recomputed from
the masked fills. Exposure lines and carved fractions remain available as
overlays.
Lasso to pick groups, then **Save selection** — the second cell reloads that
file and re-derives the passing wallets as a DataFrame.

In [17]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

from wallet_filter_explorer import open_wallet_filter_explorer


explorer = open_wallet_filter_explorer(
    wallet_vol_train_full,
    wallet_vol_train,
    variant=variant,
    eligible_wallets=eligible_wallets,
    selected_wallets=copyable_wallets,
    fills=df_full_buy_full,
    split_date=TRAIN_END,
    exposure_fracs=EXPOSURE_FRACS,
    flow_price_lo=FLOW_PRICE_LO,
    flow_price_hi=FLOW_PRICE_HI,
    flow_share_thr=FLOW_SHARE_THR,
    flow_strict=FLOW_STRICT,
    flow_carve_frac=FLOW_CARVE_FRAC,
    out_dir=WORKSPACE_DIR,
)


Wallet filter explorer (sports) running at http://127.0.0.1:53911/
  save from the page, then re-run the load cell; stop with explorer.stop()


In [18]:
import importlib

import wallet_filter_explorer as _wfe

importlib.reload(_wfe)  # pick up module edits without restarting the kernel

selection = _wfe.load_selection(explorer.selection_path)
if selection is None:
    print(f"Nothing saved yet — use 'Save selection' in {explorer.url}")
else:
    # Saved selections can become stale when the stats frames change (e.g.
    # after a data refresh). Keep the wallets that still exist and warn about
    # the rest instead of failing.
    _available = set(wallet_vol_train_full["wallet"]) | set(wallet_vol_train["wallet"])
    _missing = sorted(set(selection.get("passing") or []) - _available)
    if _missing:
        print(
            f"Warning: ignoring {len(_missing)} stale saved wallet(s) that are "
            f"missing from the current stats frames: {_missing[:5]}"
            + (" ..." if len(_missing) > 5 else "")
            + " — re-save the selection to refresh it."
        )
        selection = {
            **selection,
            "passing": [w for w in selection.get("passing") or [] if w in _available],
            "picked": [w for w in selection.get("picked") or [] if w in _available],
        }

    explored_wallets = _wfe.apply_selection_filters(
        selection,
        wallet_vol_train_full,
        wallet_vol_train,
        fills=df_full_buy_full,
    )
    print(
        f"passing={len(explored_wallets)} "
        f"picked={int(explored_wallets['picked'].sum())} "
        f"copyable_pnl={explored_wallets['copyable_pnl__carved'].sum():,.0f}"
    )
    display(explored_wallets)


passing=38 picked=0 copyable_pnl=421,879


,wallet,num_buckets__full,trade_count__full,total_notional__full,total_pnl__full,copyable_pnl__full,num_markets__full,median_roi__full,average_roi__full,max_drawdown__full,...,max_copyable_drawdown_to_copyable_pnl__carved,return__carved,max_drawdown_all__carved,total_pnl_all__carved,max_drawdown_all_to_pnl_all__carved,copyable_pnl_factor__carved,copyable_roi__carved,buy_copyable_roi__carved,median_dt_days_ago__carved,picked
0,0x0d18e30e252008d83f51dae00d36873ddd9e24a6,455,3285,2509016.02,429606.94,143544.92,52,0.09,-0.03,57992.76,...,0.00,0.24,0.00,95980.33,0.00,0.23,0.05,0.27,112.27,False
1,0x13e4bf768f4c2005af84d9f23c670ac0bc788b89,1822,5662,278901.05,25567.20,9332.98,1069,0.11,0.13,7754.17,...,0.32,0.22,686.24,3492.07,0.20,0.93,0.09,0.29,116.64,False
2,0x1f7105a18d9f36aef7c83e5df210e57cf487b2e4,1157,6634,5193534.70,946715.61,423944.44,684,0.09,0.28,95313.47,...,0.14,0.10,45259.00,161734.04,0.28,0.56,0.04,0.27,68.88,False
3,0x22761026b64f1f31e9d0ca7318c86922c454efd0,260,573,95881.48,31608.77,2979.97,34,0.45,0.17,6250.82,...,0.00,0.11,5061.59,2110.35,2.40,0.64,0.26,0.58,117.40,False
4,0x24ae647bd3d6fec6ec54e3e02606a0a260804c80,733,2559,410418.91,107159.38,27546.96,469,0.10,0.26,10379.85,...,0.00,0.22,1837.15,7041.68,0.26,0.85,0.38,0.32,115.67,False
5,0x25e21e9c4e469d26aaa5701948ff8923d060db79,185,194,531323.32,31760.11,11428.30,128,0.11,0.06,4674.94,...,0.00,0.07,1380.19,7482.99,0.18,0.35,0.02,0.05,205.43,False
6,0x2a8a536cb3448f6e0c94d4a3ed7ede7855c1cb94,919,5226,314476.95,65270.09,34743.31,377,0.11,0.15,6026.11,...,0.37,0.23,3737.97,8910.98,0.42,0.47,0.10,0.26,116.67,False
7,0x2d462b29127b919ae4c085e03be44ae7077e3e2d,1932,3724,238490.31,24746.47,-869.92,1076,-1.00,0.32,3498.35,...,0.00,0.06,1596.46,1.62,984.20,0.49,0.03,0.08,112.65,False
8,0x2e617f2cb474acaba820030e4819fe92148519bd,514,1037,186795.15,51336.67,31252.80,229,0.01,-0.01,8792.54,...,0.00,0.40,0.00,9432.14,0.00,1.00,0.48,0.40,103.62,False
9,0x2f2b6e3176e4914ccc79111dd9c704904d390f9a,4493,6357,848374.99,50357.62,11878.51,3300,0.19,0.03,11106.76,...,0.44,0.09,5284.45,6721.22,0.79,0.45,-0.05,0.18,234.91,False


## Short-horizon contrarian flow filter (post-selection overlay)

Per-fill overlay applied **after** wallet cohort selection. Does not change which
wallets are selected; filters which individual BUY fills are copied in the
backtest/exposure section.

**Feature** — for each BUY fill, scan all raw trades in the same condition during
the prior 60 seconds and compute:

`flow_usdc_share_60 = (usdc_buyT − usdc_sellT + usdc_sellC − usdc_buyC) / usdc_all_60`

Negative values = net trade flow against the bought token (more USDC exited than
entered T exposure). We lack maker/taker flags, so this measures net transaction
direction, not aggressor direction.

**Two variants:**
- `nonstrict`: price ∈ [`FLOW_PRICE_LO`, `FLOW_PRICE_HI`] AND `flow_usdc_share_60 ≤ FLOW_SHARE_THR`
- `strict`: nonstrict AND `price_chg_last_own > 0` (fill price above last T trade)

The explorer above exposes the price band, share threshold, strict flag and
exposure-carve fraction. When a selection is saved, cell 23 loads it and the next
cell overrides the `FLOW_*` config defaults, so every downstream plot follows the
tuned filter. Both nonstrict and strict variants are added to the regular plots.

**Caveat:** the signal is heterogeneous by sport (weak/negative on Soccer in test).


In [19]:
# ── Resolve effective flow filter (explorer save overrides config) ────────
# The flow-filter explorer writes price band / share threshold / strict flag /
# carve fraction into wallet_filter_selection_*.json. Use those when present so
# every downstream plot reflects the tuned filter.
_sel = globals().get("selection")
_sel_flow = {} if _sel is None else (_sel.get("flow") or {})

FLOW_PRICE_LO = float(_sel_flow.get("price_lo", FLOW_PRICE_LO))
FLOW_PRICE_HI = float(_sel_flow.get("price_hi", FLOW_PRICE_HI))
FLOW_SHARE_THR = float(_sel_flow.get("share_thr", FLOW_SHARE_THR))
FLOW_STRICT = bool(_sel_flow.get("strict", FLOW_STRICT))
FLOW_CARVE_FRAC = float(_sel["carveFrac"])

selected_wallets_flow = set(wallet_cohorts["predicting_group"]["wallet"])
_carve_txt = "none" if FLOW_CARVE_FRAC is None else f"{FLOW_CARVE_FRAC:g}"
print(
    f"Flow filter: price {FLOW_PRICE_LO:g}-{FLOW_PRICE_HI:g}, "
    f"share <= {FLOW_SHARE_THR:g}, strict={FLOW_STRICT}, "
    f"carve={_carve_txt}, wallets={len(selected_wallets_flow)}"
)


def _flow_masks(frame: pd.DataFrame) -> tuple[pd.Series, pd.Series]:
    """BUY masks for the nonstrict / strict contrarian flow filter."""
    share = frame["flow_usdc_share_60"]
    is_buy = frame["side"] == "BUY"
    band = frame["price"].between(FLOW_PRICE_LO, FLOW_PRICE_HI)
    signal = share <= FLOW_SHARE_THR
    nonstrict = (is_buy & band & signal).fillna(False)
    strict = (nonstrict & (frame["price_chg_last_own"] > 0)).fillna(False)
    return nonstrict, strict


Flow filter: price 0.2-0.5, share <= -0.1, strict=False, carve=1, wallets=12


In [20]:
# ── Define flow filter variants (carved + uncarved BUY streams) ──────────
flow_fills_uncarved = df_full_buy_full.copy()
flow_fills_uncarved["flow_usdc_share_60"] = (
    flow_fills_uncarved["flow_own_usdc_60"]
    / flow_fills_uncarved["usdc_all_60"].replace(0, np.nan)
)
if "flow_usdc_share_60" not in df_full.columns:
    df_full["flow_usdc_share_60"] = (
        df_full["flow_own_usdc_60"] / df_full["usdc_all_60"].replace(0, np.nan)
    )

for _frame in (df_full, flow_fills_uncarved):
    _ns, _st = _flow_masks(_frame)
    _frame["flow_pass_nonstrict"] = _ns
    _frame["flow_pass_strict"] = _st

# ── Summary by split ───────────────────────────────────────────────────────
print("Flow filter pass rates (selected wallets, BUY fills):")
print(f"  {'stream':10s} {'split':5s} {'n':>10s}  {'nonstrict':>12s}  {'strict':>12s}")

for _label, _frame in [("carved", df_full), ("uncarved", flow_fills_uncarved)]:
    _selected = _frame["wallet"].isin(selected_wallets_flow)
    for _split, _split_mask in [
        ("train", _frame["is_train"]),
        ("test", ~_frame["is_train"]),
    ]:
        _mask = _frame["side"].eq("BUY") & _selected & _split_mask
        _n = int(_mask.sum())
        if not _n:
            continue
        _ns = int(_frame.loc[_mask, "flow_pass_nonstrict"].sum())
        _st = int(_frame.loc[_mask, "flow_pass_strict"].sum())
        print(
            f"  {_label:10s} {_split:5s} {_n:>10,}  "
            f"{_ns:>7,} ({_ns / _n:5.1%})  {_st:>7,} ({_st / _n:5.1%})"
        )


Flow filter pass rates (selected wallets, BUY fills):
  stream     split          n     nonstrict        strict
  carved     train      8,155    2,162 (26.5%)      409 ( 5.0%)
  carved     test       3,776    1,034 (27.4%)      307 ( 8.1%)
  uncarved   train     73,772   16,008 (21.7%)    4,126 ( 5.6%)
  uncarved   test      33,294    9,807 (29.5%)    2,236 ( 6.7%)


In [21]:
# ── Flow-limited copyable PnL variants ─────────────────────────────────────
# Create PnL columns that zero out fills failing the flow filter. Register
# them so the existing exposure / PnL-over-time plots pick them up
# automatically.
#
# Variants per flow filter (nonstrict / strict):
#   copyable_pnl_flow[_strict]               — base copyable PnL × flow mask
# Each gets a matching _exposure column.

for _variant, _mask_col in [
    ("flow", "flow_pass_nonstrict"),
    ("flow_strict", "flow_pass_strict"),
]:
    _mask = df_full[_mask_col].fillna(False)
    _mask_f = _mask.astype(float)

    # ── Flow-limited copyable PnL (base × flow mask) ──────────────────────
    _pnl = f"copyable_pnl_{_variant}"
    _not = f"copyable_notional_{_variant}"
    _exp = f"{_pnl}_exposure"

    df_full[_pnl] = np.where(_mask, df_full["copyable_pnl"], 0.0)
    df_full[_not] = np.where(_mask, df_full["copyable_notional"], 0.0)

    _base_exp = np.where(
        df_full["copyable_qty_5m_100"] > 0,
        df_full["price"] * df_full["copyable_qty_5m_100"],
        np.nan,
    )
    # 1e-9 (not 0.0): downstream scaling does replace(0, nan) before dividing;
    # 0 * nan = nan would break cumsum. Tiny positive keeps scale=1, pnl=0.
    df_full[_exp] = np.where(_mask, _base_exp, 1e-9)

    if _pnl not in COPYABLE_PNL_COLS:
        COPYABLE_PNL_COLS.append(_pnl)



print(f"COPYABLE_PNL_COLS: {COPYABLE_PNL_COLS}")

# ── Quick summary on carved test stream ─────────────────────────────────────
_tb = (~df_full["is_train"]) & df_full["wallet"].isin(selected_wallets_flow)
_flow_notionals = {
    "copyable_pnl_flow": "copyable_notional_flow",
    "copyable_pnl_flow_strict": "copyable_notional_flow_strict",
}
for _label, _col in [
    ("base", "copyable_pnl"),
    ("nonstrict", "copyable_pnl_flow"),
    ("strict", "copyable_pnl_flow_strict"),
]:
    _p = df_full.loc[_tb, _col].sum()
    _n_col = _flow_notionals.get(_col, "copyable_notional")
    _n = df_full.loc[_tb, _n_col].sum()
    _r = _p / _n if _n and _n > 0 else np.nan
    print(f"  {_label:12s}  pnl={_p:>+12,.0f}  roi={_r:>+.4f}")


COPYABLE_PNL_COLS: ['copyable_pnl', 'copyable_pnl_flow', 'copyable_pnl_flow_strict']
  base          pnl=      +8,041  roi=+0.0137
  nonstrict     pnl=      +4,445  roi=+0.0568
  strict        pnl=      +1,802  roi=+0.0631


## Size-share filter (post-selection overlay)

Per-fill overlay applied **after** wallet cohort selection. Does not change which
wallets are selected; it filters which individual BUY fills are copied. The
overlay is fitted and applied to the **predicting group** (the stage2 cohort);
the copyable group keeps its unfiltered stream.

**Feature** — `size_share = usdc_amount / usdc_all_60` compares the wallet's
current BUY notional to the total notional traded in the same market over the
prior 60 seconds. Both inputs are observable when the fill is seen, so the
filter is ex-ante.

- **Small** `size_share` — the wallet's buy is small relative to recent market
  activity; a copy is less likely to move the price and more likely to fill well.
- **Large** `size_share` — the wallet dominates the recent tape; copying is more
  exposed to impact and execution risk.

**Rule** — price ∈ [`SIZE_SHARE_PRICE_LO`, `SIZE_SHARE_PRICE_HI`] AND
`size_share` ≤ T, where T is the `SIZE_SHARE_QUANTILE` quantile of the train
BUY fills of the predicting-group wallets.


In [22]:
# ── Resolve the size-share threshold on the train carved stream ───────────
selected_wallets_size = set(wallet_cohorts["predicting_group"]["wallet"])

_size_train = df_full[
    df_full["is_train"]
    & (df_full["side"] == "BUY")
    & (df_full["wallet"].isin(selected_wallets_size))
]
SIZE_SHARE_THR = _size_train["size_share"].quantile(SIZE_SHARE_QUANTILE)
del _size_train

print(
    f"Size filter: price {SIZE_SHARE_PRICE_LO:g}-{SIZE_SHARE_PRICE_HI:g}, "
    f"size_share <= {SIZE_SHARE_THR:.4f} (train q{SIZE_SHARE_QUANTILE:.0%}), "
    f"predicting-group wallets={len(selected_wallets_size)}"
)


def _size_masks(frame: pd.DataFrame) -> pd.Series:
    """BUY mask for the ex-ante size-share filter."""
    is_buy = frame["side"] == "BUY"
    band = frame["price"].between(SIZE_SHARE_PRICE_LO, SIZE_SHARE_PRICE_HI)
    signal = frame["size_share"] <= SIZE_SHARE_THR
    return (is_buy & band & signal).fillna(False)


df_full["size_pass"] = _size_masks(df_full)

if "size_share" not in flow_fills_uncarved.columns:
    flow_fills_uncarved["size_share"] = (
        flow_fills_uncarved["notional"].astype(float)
        / flow_fills_uncarved["usdc_all_60"].astype(float).replace(0, np.nan)
    )
flow_fills_uncarved["size_pass"] = _size_masks(flow_fills_uncarved)

# ── Pass-rate summary (selected wallets, BUY fills) ───────────────────────
print("Size filter pass rates (selected wallets, BUY fills):")
for _label, _frame in [("carved", df_full), ("uncarved", flow_fills_uncarved)]:
    _selected = _frame["wallet"].isin(selected_wallets_size)
    for _split, _split_mask in [
        ("train", _frame["is_train"]),
        ("test", ~_frame["is_train"]),
    ]:
        _mask = _selected & _split_mask
        _n = int(_mask.sum())
        if not _n:
            continue
        _k = int(_frame.loc[_mask, "size_pass"].sum())
        print(
            f"  {_label:10s} {_split:5s} {_n:>10,}  "
            f"{_k:>7,} ({_k / _n:5.1%})"
        )


Size filter: price 0.3-0.75, size_share <= 0.0174 (train q50%), predicting-group wallets=12
Size filter pass rates (selected wallets, BUY fills):
  carved     train      8,155    2,747 (33.7%)
  carved     test       3,776    1,080 (28.6%)
  uncarved   train     73,772   24,052 (32.6%)
  uncarved   test      33,294   11,028 (33.1%)


In [23]:
# ── Size-limited copyable PnL variant ─────────────────────────────────────
# Mirrors the flow overlay: zero out fills failing the size filter and
# register the variant so downstream plots / stats pick it up automatically.
_mask = (
    df_full["size_pass"].fillna(False)
    & df_full["wallet"].isin(selected_wallets_size)
)

df_full["copyable_pnl_size"] = np.where(_mask, df_full["copyable_pnl"], 0.0)
df_full["copyable_notional_size"] = np.where(
    _mask, df_full["copyable_notional"], 0.0
)

_base_exp = np.where(
    df_full["copyable_qty_5m_100"] > 0,
    df_full["price"] * df_full["copyable_qty_5m_100"],
    np.nan,
)
# 1e-9 (not 0.0): downstream scaling does replace(0, nan) before dividing;
# 0 * nan = nan would break cumsum. Tiny positive keeps scale=1, pnl=0.
df_full["copyable_pnl_size_exposure"] = np.where(_mask, _base_exp, 1e-9)

if "copyable_pnl_size" not in COPYABLE_PNL_COLS:
    COPYABLE_PNL_COLS.append("copyable_pnl_size")

print(f"COPYABLE_PNL_COLS: {COPYABLE_PNL_COLS}")

# ── Quick summary on the carved test stream ────────────────────────────────
_tb = (~df_full["is_train"]) & df_full["wallet"].isin(selected_wallets_size)
_sub = df_full.loc[_tb]
_base_fills = int((_sub["copyable_qty_5m_100"] > 0).sum())
_size_summary = []
for _label, _pnl_col, _not_col in [
    ("base", "copyable_pnl", "copyable_notional"),
    ("size", "copyable_pnl_size", "copyable_notional_size"),
]:
    _p = _sub[_pnl_col].sum()
    _n = _sub[_not_col].sum()
    if _label == "base":
        _fills = _base_fills
    else:
        _fills = int((_sub["size_pass"] & (_sub["copyable_qty_5m_100"] > 0)).sum())
    _size_summary.append({
        "variant": _label,
        "fills": _fills,
        "retention": _fills / _base_fills if _base_fills else float("nan"),
        "copyable_pnl": _p,
        "copyable_roi": _p / _n if _n and _n > 0 else float("nan"),
    })
display(pd.DataFrame(_size_summary).round(4))


COPYABLE_PNL_COLS: ['copyable_pnl', 'copyable_pnl_flow', 'copyable_pnl_flow_strict', 'copyable_pnl_size']


,variant,fills,retention,copyable_pnl,copyable_roi
0,base,3357,1.00,8041.26,0.01
1,size,996,0.30,18559.60,0.26


## Flow-selection filter (port of `stage1_flow_selection`)

Ports the chosen ex-ante market situation from `stage1_flow_selection` exactly and
registers it as the `copyable_pnl_price_size` variant, so every downstream stat /
plot picks it up automatically.

Pipeline (identical code path to `stage1_flow_selection`):

1. candidate universe via `build_sports_candidate_universe` (original
   strategy-selection 2-day lead window, `variant` as resolved above)
2. uncarved BUY fills of those wallets in `primary_tag ∈ TAGS` conditions, lead
   `FLOWSEL_MIN_LEAD_DAYS`-`FLOWSEL_MAX_LEAD_DAYS` days before resolution,
   `copyable_notional > 0`
3. fit / val / test split by market resolution date (`FLOWSEL_FIT_END` /
   `FLOWSEL_VAL_END`)
4. fit-profitability screen, then `size_share` threshold = fit
   q`FLOWSEL_SIZE_SHARE_QUANTILE`
5. chosen mask: price ∈ `FLOWSEL_PRICE_BAND` AND `size_share` ≤ threshold
6. wallet screen: profitable in fit AND val under the mask
   (`FLOWSEL_MIN_FIT/VAL_WALLET_FILLS` min fills)

The eval table below must reproduce `stage1_flow_selection`'s saved numbers. The
final per-fill mask is then joined onto `df_full` by
`(wallet, condition_id, token_id, tx_hash)`; totals restricted to `df_full`
additionally reflect this notebook's exposure carve and market filter, which the
uncarved reference table does not.

**Why the `df_full` curves are much smaller than the reference:** the exposure
carve (frac 0.2) keeps only high deployed-capital fills, while the
flow-selection mask keeps *small* `size_share` fills — the two are nearly
anti-correlated, so ~97% of the final-mask fills are carved away (the largest
flow-selected wallet retains zero carved test fills). The market filter and
basic wallet eligibility remove nothing; the per-cohort table and every
`df_full`-based plot reflect the carve. The plot at the end of this section
shows the uncarved reference (matching `stage1_flow_selection`, test cpnl
105,843) next to the same mask restricted to carved `df_full`.

In [24]:
# ── Build the flow-selection stream (exact stage1_flow_selection replica) ──
from signal_lab.candidate_universe import build_sports_candidate_universe

_fs_mdf = load_markets_processed()
_fs_mdf = _fs_mdf[_fs_mdf["primary_tag"].isin(TAGS)].copy()
flowsel_conditions = set(_fs_mdf["condition_id"])
del _fs_mdf

_, _, flowsel_candidate_wallets, _, _ = build_sports_candidate_universe(
    TRADES_DIR,
    max_lead_days=FLOWSEL_CANDIDATE_MAX_LEAD_DAYS,
    variant=variant,
)
flowsel_wallets = set(map(str, flowsel_candidate_wallets))
print(f"flow-selection candidate wallets: {len(flowsel_wallets):,}")

_fs_cols = [
    "wallet", "condition_id", "token_id", "tx_hash", "dt", "side",
    "avg_price", "trade_value_usdc", "copyable_pnl",
    "last_condition_trade_ts", "copyable_qty_5m_100", "usdc_all_60",
]
fs = (
    ds.dataset(TRADES_DIR, format="parquet")
    .to_table(columns=_fs_cols, filter=ds.field("wallet").isin(list(flowsel_wallets)))
    .to_pandas()
)
fs["wallet"] = fs["wallet"].astype(str)

flowsel_key_cols = ["wallet", "condition_id", "token_id", "tx_hash"]
assert not fs.duplicated(subset=flowsel_key_cols).any(), "flow-selection join key not unique"

fs = fs[fs["side"] == "BUY"].copy()
fs = fs[fs["condition_id"].isin(flowsel_conditions)].copy()
fs = fs.rename(columns={"avg_price": "price", "trade_value_usdc": "usdc_amount"})

_fs_lead = pd.to_datetime(fs["last_condition_trade_ts"], utc=True, errors="coerce") - fs["dt"]
fs = fs[
    (_fs_lead <= pd.Timedelta(days=FLOWSEL_MAX_LEAD_DAYS))
    & (_fs_lead >= pd.Timedelta(days=FLOWSEL_MIN_LEAD_DAYS))
].copy()
fs["dt"] = pd.to_datetime(fs["dt"], utc=True)
fs["resolution_dt"] = pd.to_datetime(fs["last_condition_trade_ts"], utc=True)

# Evaluation-only notional: how much of the fill could actually be copied.
fs["copyable_notional"] = (
    fs["price"].astype(float)
    * fs["copyable_qty_5m_100"].astype(float).clip(lower=0)
)
fs = fs[fs["copyable_notional"] > 0].copy()

# Ex-ante signal feature: current trade size relative to prior-60s volume.
fs["size_share"] = (
    fs["usdc_amount"].astype(float)
    / fs["usdc_all_60"].astype(float).replace(0, np.nan)
)

fs["is_fit"] = fs["resolution_dt"] <= FLOWSEL_FIT_END
fs["is_val"] = (fs["resolution_dt"] > FLOWSEL_FIT_END) & (fs["resolution_dt"] <= FLOWSEL_VAL_END)
fs["is_test"] = fs["resolution_dt"] > FLOWSEL_VAL_END

# Fit-profitability screen — applied BEFORE the threshold fit, exactly as in
# stage1_flow_selection.
_fs_fit_wallets = fs[fs["is_fit"]].groupby("wallet").agg(
    cpnl=("copyable_pnl", "sum"),
    fills=("copyable_pnl", "size"),
)
fs = fs[
    fs["wallet"].isin(
        _fs_fit_wallets.index[
            (_fs_fit_wallets["cpnl"] > 0)
            & (_fs_fit_wallets["fills"] >= FLOWSEL_MIN_FIT_WALLET_FILLS)
        ]
    )
].copy()
print(f"wallets after fit-profitability screen: {fs['wallet'].nunique():,}; fills: {len(fs):,}")

flowsel_period_cols = {"fit": "is_fit", "val": "is_val", "test": "is_test"}
flowsel_baseline_fills = {p: int(fs[c].sum()) for p, c in flowsel_period_cols.items()}
FLOWSEL_THR = fs[fs["is_fit"]]["size_share"].quantile(FLOWSEL_SIZE_SHARE_QUANTILE)
print(f"baseline fills: {flowsel_baseline_fills}")
print(f"size_share threshold (fit q{FLOWSEL_SIZE_SHARE_QUANTILE:.0%}): {FLOWSEL_THR:.4f}")

/Users/vobornij/projects/polymarket/src/polymarket_analysis/wallet_selection/volatility.py:190: FutureWarning:

The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior

/Users/vobornij/projects/polymarket/src/polymarket_analysis/wallet_selection/volatility.py:190: FutureWarning:

The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior

/Users/vobornij/projects/polymarket/src/polymarket_analysis/wallet_selection/volatility.py:190: FutureWarning:

The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior

/Users/vobornij/projects/polymarket/src/polymarket_analysis/wallet_selection/volat

flow-selection candidate wallets: 35
wallets after fit-profitability screen: 31; fills: 249,791
baseline fills: {'fit': 110508, 'val': 51525, 'test': 87758}
size_share threshold (fit q50%): 0.0256


In [25]:
# ── Chosen mask, wallet screen, and eval table (stage1_flow_selection replica) ──
flowsel_mask = (
    fs["price"].between(FLOWSEL_PRICE_BAND[0], FLOWSEL_PRICE_BAND[1])
    & (fs["size_share"] <= FLOWSEL_THR)
).fillna(False)


def _flowsel_screen_stats(mask, period):
    return fs[mask & fs[flowsel_period_cols[period]]].groupby("wallet").agg(
        cpnl=("copyable_pnl", "sum"),
        fills=("copyable_pnl", "size"),
    )


_fs_fit = _flowsel_screen_stats(flowsel_mask, "fit")
_fs_val = _flowsel_screen_stats(flowsel_mask, "val")
_fs_fit_ok = _fs_fit.index[
    (_fs_fit["cpnl"] > 0) & (_fs_fit["fills"] >= FLOWSEL_MIN_FIT_WALLET_FILLS)
]
_fs_val_ok = _fs_val.index[
    (_fs_val["cpnl"] > 0) & (_fs_val["fills"] >= FLOWSEL_MIN_VAL_WALLET_FILLS)
]
flowsel_selected_wallets = set(_fs_fit_ok.intersection(_fs_val_ok))
flowsel_final_mask = flowsel_mask & fs["wallet"].isin(flowsel_selected_wallets)

print(f"price band: [{FLOWSEL_PRICE_BAND[0]:g}, {FLOWSEL_PRICE_BAND[1]:g}]")
print(f"size_share <= {FLOWSEL_THR:.4f} (fit q{FLOWSEL_SIZE_SHARE_QUANTILE:.0%})")
print(f"selected wallets: {len(flowsel_selected_wallets):,}")


def _flowsel_metrics(mask, period):
    sub = fs[mask & fs[flowsel_period_cols[period]]]
    base_n = flowsel_baseline_fills[period]
    if sub.empty:
        return {
            "fills": 0,
            "retention": 0.0,
            "roi": np.nan,
            "sharpe": np.nan,
            "max_dd": np.nan,
            "pos_day": np.nan,
            "days": 0,
            "cpnl": 0.0,
            "notional": 0.0,
        }
    daily = sub.groupby(sub["dt"].dt.floor("D"))["copyable_pnl"].sum()
    sharpe = daily.mean() / daily.std() * np.sqrt(365) if daily.std() > 0 else np.nan
    cumulative = daily.cumsum()
    return {
        "fills": len(sub),
        "retention": len(sub) / base_n if base_n else np.nan,
        "roi": float(sub["copyable_pnl"].sum() / sub["copyable_notional"].sum()),
        "sharpe": float(sharpe),
        "max_dd": float((cumulative.cummax() - cumulative).max()),
        "pos_day": float((daily > 0).mean()),
        "days": len(daily),
        "cpnl": float(sub["copyable_pnl"].sum()),
        "notional": float(sub["copyable_notional"].sum()),
    }


_flowsel_rows = []
for _label, _mask in [
    ("baseline (all candidate wallets)", pd.Series(True, index=fs.index)),
    ("chosen filter", flowsel_mask),
    ("chosen filter + wallet screen", flowsel_final_mask),
]:
    for _period in ("fit", "val", "test"):
        _flowsel_rows.append(
            {"variant": _label, "period": _period, **_flowsel_metrics(_mask, _period)}
        )
display(pd.DataFrame(_flowsel_rows).round(3))

price band: [0.3, 0.75]
size_share <= 0.0256 (fit q50%)
selected wallets: 15


,variant,period,fills,retention,roi,sharpe,max_dd,pos_day,days,cpnl,notional
0,baseline (all candidate wallets),fit,110508,1.00,0.07,2.75,217247.66,0.64,224,1107400.75,15303606.16
1,baseline (all candidate wallets),val,51525,1.00,0.08,7.21,70466.80,0.59,37,618470.56,7376803.20
2,baseline (all candidate wallets),test,87758,1.00,0.00,0.28,218135.64,0.43,65,33616.88,13591455.97
3,chosen filter,fit,33299,0.30,0.12,3.07,38350.57,0.56,202,224518.49,1952290.25
4,chosen filter,val,14250,0.28,0.12,6.20,11848.74,0.64,36,104922.43,866271.01
5,chosen filter,test,23801,0.27,0.07,5.84,25226.08,0.70,63,100315.34,1331770.91
6,chosen filter + wallet screen,fit,27027,0.24,0.12,3.12,40682.85,0.60,183,214111.75,1754556.11
7,chosen filter + wallet screen,val,12288,0.24,0.12,5.93,13996.58,0.63,35,100429.67,818263.63
8,chosen filter + wallet screen,test,20229,0.23,0.08,6.33,23044.98,0.73,63,105843.00,1255783.27


In [26]:
# ── Register copyable_pnl_price_size on df_full (join by fill key) ────────
_flowsel_keys = fs.loc[flowsel_final_mask, flowsel_key_cols].assign(_pass=True)
_price_size_pass = (
    df_full.reset_index()
    .merge(_flowsel_keys, on=flowsel_key_cols, how="left")
    .set_index("index")["_pass"]
    .fillna(False)
)
df_full["price_size_pass"] = _price_size_pass.to_numpy()

_mask = df_full["price_size_pass"]
df_full["copyable_pnl_price_size"] = np.where(_mask, df_full["copyable_pnl"], 0.0)
df_full["copyable_notional_price_size"] = np.where(
    _mask, df_full["copyable_notional"], 0.0
)
_base_exp = np.where(
    df_full["copyable_qty_5m_100"] > 0,
    df_full["price"] * df_full["copyable_qty_5m_100"],
    np.nan,
)
# 1e-9 (not 0.0): downstream scaling does replace(0, nan) before dividing;
# 0 * nan = nan would break cumsum. Tiny positive keeps scale=1, pnl=0.
df_full["copyable_pnl_price_size_exposure"] = np.where(_mask, _base_exp, 1e-9)

if "copyable_pnl_price_size" not in COPYABLE_PNL_COLS:
    COPYABLE_PNL_COLS.append("copyable_pnl_price_size")
print(f"COPYABLE_PNL_COLS: {COPYABLE_PNL_COLS}")

_dfps_n = int(df_full["price_size_pass"].sum())
_fs_n = int(flowsel_final_mask.sum())
print(
    f"restricted to carved df_full: {_dfps_n:,} of {_fs_n:,} final-mask fills; "
    f"test cpnl {df_full.loc[~df_full['is_train'], 'copyable_pnl_price_size'].sum():,.0f} "
    "(exposure carve removes most small-size_share fills — see the plot below)"
)

print(
    "uncarved flow-selection cpnl (reference, matches stage1_flow_selection): "
    f"fit {fs.loc[flowsel_final_mask & fs['is_fit'], 'copyable_pnl'].sum():,.0f} / "
    f"val {fs.loc[flowsel_final_mask & fs['is_val'], 'copyable_pnl'].sum():,.0f} / "
    f"test {fs.loc[flowsel_final_mask & fs['is_test'], 'copyable_pnl'].sum():,.0f}"
)

_summary_rows = []
for _group, _cohort in [
    ("copyable", "copyable_group"),
    ("predicting", "predicting_group"),
]:
    _wallets_c = set(wallet_cohorts[_cohort]["wallet"])
    _sub = df_full[(~df_full["is_train"]) & (df_full["wallet"].isin(_wallets_c))]
    _base_fills = int((_sub["copyable_qty_5m_100"] > 0).sum())
    for _label, _pnl_col, _not_col in [
        ("base", "copyable_pnl", "copyable_notional"),
        ("price_size", "copyable_pnl_price_size", "copyable_notional_price_size"),
    ]:
        _p = _sub[_pnl_col].sum()
        _n = _sub[_not_col].sum()
        _fills = (
            _base_fills
            if _label == "base"
            else int((_sub["price_size_pass"] & (_sub["copyable_qty_5m_100"] > 0)).sum())
        )
        _summary_rows.append(
            {
                "group": _group,
                "variant": _label,
                "fills": _fills,
                "retention": _fills / _base_fills if _base_fills else float("nan"),
                "copyable_pnl": _p,
                "copyable_roi": _p / _n if _n and _n > 0 else float("nan"),
            }
        )
display(pd.DataFrame(_summary_rows).round(4))

COPYABLE_PNL_COLS: ['copyable_pnl', 'copyable_pnl_flow', 'copyable_pnl_flow_strict', 'copyable_pnl_size', 'copyable_pnl_price_size']
restricted to carved df_full: 1,833 of 59,544 final-mask fills; test cpnl 10,399 (exposure carve removes most small-size_share fills — see the plot below)
uncarved flow-selection cpnl (reference, matches stage1_flow_selection): fit 214,112 / val 100,430 / test 105,843


/var/folders/j8/0dbnwk8n6m933m843h7hb88w0000gn/T/ipykernel_45045/3960173590.py:7: FutureWarning:

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`



,group,variant,fills,retention,copyable_pnl,copyable_roi
0,copyable,base,948,1.00,78085.37,0.07
1,copyable,price_size,21,0.02,1456.55,0.17
2,predicting,base,3357,1.00,8041.26,0.01
3,predicting,price_size,517,0.15,10885.58,0.14


In [27]:
# ── Cumulative copyable PnL: flow-selection reference vs carved df_full ────
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def _flowsel_daily(mask, test_only=False):
    sub = fs[mask]
    if test_only:
        sub = sub[sub["is_test"]]
    return sub.groupby(sub["dt"].dt.floor("D"))["copyable_pnl"].sum()


_dfps_test = (
    df_full.loc[~df_full["is_train"]]
    .assign(_d=lambda d: d["dt"].dt.floor("D"))
    .groupby("_d")["copyable_pnl_price_size"]
    .sum()
)

fig = make_subplots(
    rows=2,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=("Full timeline (fit | val | test)", "Test period only"),
)

for _name, _mask, _color in [
    ("baseline (all candidate wallets)", pd.Series(True, index=fs.index), "#636efa"),
    ("chosen filter", flowsel_mask, "#ff7f0e"),
    ("chosen filter + wallet screen", flowsel_final_mask, "#2ca02c"),
]:
    _curve = _flowsel_daily(_mask).cumsum()
    fig.add_trace(
        go.Scatter(
            x=_curve.index, y=_curve.values, mode="lines", name=_name,
            line=dict(color=_color, width=2),
        ),
        row=1, col=1,
    )

_curve = _flowsel_daily(flowsel_final_mask, test_only=True).cumsum()
fig.add_trace(
    go.Scatter(
        x=_curve.index, y=_curve.values, mode="lines",
        name="chosen filter + wallet screen (uncarved — matches stage1_flow_selection)",
        line=dict(color="#2ca02c", width=2),
    ),
    row=2, col=1,
)
_curve = _dfps_test.cumsum()
fig.add_trace(
    go.Scatter(
        x=_curve.index, y=_curve.values, mode="lines",
        name="same mask on carved df_full (all cohorts)",
        line=dict(color="#d62728", width=2, dash="dash"),
    ),
    row=2, col=1,
)

for _ts in (FLOWSEL_FIT_END, FLOWSEL_VAL_END):
    fig.add_vline(x=_ts, line=dict(color="grey", width=1, dash="dot"), row=1, col=1)
    fig.add_vline(x=_ts, line=dict(color="grey", width=1, dash="dot"), row=2, col=1)

fig.update_layout(
    template="plotly_white",
    height=700,
    title=(
        "Flow-selection filter — cumulative copyable PnL<br>"
        f"<sup>test end: uncarved reference {_flowsel_daily(flowsel_final_mask, test_only=True).sum():,.0f} "
        f"(matches stage1_flow_selection) vs carved df_full {_dfps_test.sum():,.0f}</sup>"
    ),
    yaxis_title="Cumulative copyable PnL (USDC)",
    yaxis2_title="Cumulative copyable PnL (USDC)",
)
fig.show()

In [31]:
print("TEST")
wallet_vol_test[wallet_vol_test['wallet'] == '0xc3973025163de81e367ad29c21e296e51684bb3b']

TEST


,wallet,num_buckets,trade_count,total_notional,total_pnl,copyable_pnl,num_markets,median_dt,median_roi,average_roi,...,estimated_copyable_buy_sharpe,copyable_pnl_similarity,copyable_pnl_similarity_profit,copyable_pnl_similarity_loss,copyable_pnl_similarity_20m_100,copyable_pnl_similarity_profit_20m_100,copyable_pnl_similarity_loss_20m_100,max_drawdown_to_pnl,max_copyable_drawdown_to_copyable_pnl,return
412,0xc3973025163de81e367ad29c21e296e51684bb3b,96,111,185092.96,23070.85,2388.38,69,2026-10-12 18:32:30+00:00,0.22,0.08,...,0.46,0.92,0.89,1.00,NaN,NaN,NaN,0.39,4.58,0.12


In [32]:
selected_wallets_vol = wallet_vol_test[wallet_vol_test['wallet'].isin(wallet_cohorts['predicting_group'].wallet)]
print("TEST")
selected_wallets_vol.sort_values(by='copyable_pnl', key=abs, ascending=False).head(50)

TEST


,wallet,num_buckets,trade_count,total_notional,total_pnl,copyable_pnl,num_markets,median_dt,median_roi,average_roi,...,estimated_copyable_buy_sharpe,copyable_pnl_similarity,copyable_pnl_similarity_profit,copyable_pnl_similarity_loss,copyable_pnl_similarity_20m_100,copyable_pnl_similarity_profit_20m_100,copyable_pnl_similarity_loss_20m_100,max_drawdown_to_pnl,max_copyable_drawdown_to_copyable_pnl,return
270,0x8f8883ecf250d19cef876e6be32188b27f2bca10,121,732,115621.41,11086.15,-7109.54,110,2026-10-01 19:05:00+00:00,-1.00,0.08,...,-1.19,0.84,0.81,0.92,NaN,NaN,NaN,1.62,NaN,0.10
423,0xc2494b32d65dea3a3a80712589d10bc5733f8a61,195,358,68511.33,-2492.37,-4544.79,147,2026-10-12 17:15:00+00:00,-1.00,-0.09,...,-4.40,0.59,0.46,0.70,NaN,NaN,NaN,NaN,NaN,-0.04
335,0x149a9ec31ae097faa33b2f2ae8df659889c149d0,168,393,153798.52,10053.70,4450.03,130,2026-10-07 16:47:30+00:00,0.25,0.06,...,1.31,0.96,0.95,0.97,NaN,NaN,NaN,0.73,1.49,0.07
386,0x4d1452faf4de78b990e6ec7b9d69217c1df636fd,16,91,8714.41,5646.03,3572.59,12,2026-09-16 01:17:30+00:00,0.71,0.41,...,4.64,0.83,0.86,NaN,NaN,NaN,NaN,0.16,0.06,0.65
111,0xf49614e63fb15383d4a9b717a1be03ad2410fe79,41,234,102931.25,4798.51,3545.26,34,2026-09-20 22:35:00+00:00,0.01,0.16,...,1.53,0.97,0.98,0.96,NaN,NaN,NaN,1.13,1.60,0.05
251,0x984ffef12a23e7af5e2cef8e3283e0ab6a6e9a43,184,314,115803.65,3595.48,2439.38,147,2026-09-28 13:10:00+00:00,0.36,0.04,...,1.26,0.72,0.76,0.68,NaN,NaN,NaN,1.75,1.63,0.03
412,0xc3973025163de81e367ad29c21e296e51684bb3b,96,111,185092.96,23070.85,2388.38,69,2026-10-12 18:32:30+00:00,0.22,0.08,...,0.46,0.92,0.89,1.00,NaN,NaN,NaN,0.39,4.58,0.12
143,0x98689f5983898c1da0ec7a9f0864d478f3bee82b,302,902,25122.02,2070.87,1800.88,140,2026-09-23 19:32:30+00:00,-1.00,0.05,...,1.53,0.91,0.92,0.90,NaN,NaN,NaN,0.88,0.74,0.08
117,0xc34b4901be324718c176100e350211d97b199296,269,304,251676.10,-4372.69,-1203.25,237,2026-09-09 16:50:00+00:00,0.25,-0.00,...,-0.43,0.58,0.56,0.61,NaN,NaN,NaN,NaN,NaN,-0.02
599,0x73492e9ea5e7423cc0bb10a7169a0c81972dcc3a,4,31,2754.51,-2754.51,-1035.43,4,2026-09-13 10:00:00+00:00,-1.00,-1.00,...,-10.05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.00


In [33]:
# ── Individual wallet inspection: PnL / copyable PnL variants / exposure ─────
# Per wallet: two stacked subplots (train / test) with cumulative wallet PnL,
# copyable-PnL variants (raw/unlimited) and their copy exposure (secondary
# axis). Additionally the bankroll-discipline carve variants per
# EXPOSURE_FRACS: T_{w,f} is fit on the wallet's TRAIN fills via
# wallet_capture_thresholds (smallest deployed capital capturing fraction `f`
# of the wallet's train copyable PnL; f == 1 is the raw baseline above), and
# the same thresholds carve the fills in both subplots.
import plotly.colors
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from polymarket_analysis.visualization.wallet_plots import copyable_exposure_series
from polymarket_analysis.wallet_selection.exposure import wallet_capture_thresholds

INSPECT_WALLETS = [
    # "0xc1d971eadb83e32358b828cef7a6f722770c5333",
    # "0xc3973025163de81e367ad29c21e296e51684bb3b",
    # "0xf64745161e46cbd2143f021edd92cf6bed850ccb"
    "0x08c263965a9cbec5e83ac1d6f14b9bbc957bc023"
]

PNL_TO_QTY = {
    COPYABLE_PNL_BASE: "copyable_qty_5m_100",
    **{f"copyable_pnl_{k}": cfg["qty_col"] for k, cfg in COPYABLE_VARIANTS.items()},
}
LINE_COLORS = {
    "pnl": "#1f77b4",
    **{c: col for c, col in zip(
        COPYABLE_PNL_COLS,
        ["#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#e377c2", "#8c564b", "#17becf", "#bcbd22"],
    )}
}

CARVE_FRACS = sorted({f for f in EXPOSURE_FRACS if f < 1}, reverse=True)
FRAC_COLORS = {f: col for f, col in zip(CARVE_FRACS, plotly.colors.qualitative.Dark2)}

for wallet in INSPECT_WALLETS:
    w = df_full[df_full["wallet"] == wallet].copy()
    if w.empty:
        print(f"{wallet}: no fills in df_full, skipping")
        continue

    # per-fill copy exposure (price × copyable qty)
    for pnl_col, qty_col in PNL_TO_QTY.items():
        w[f"{pnl_col}_exposure"] = np.where(
            w[qty_col] > 0,
            w["price"] * w[qty_col],
            np.nan,
        )
    w["deployed_capital"] = w["position"] * w["price"]

    # carve thresholds T_{w,f}: fit on the wallet's train fills only
    fit = w[w["is_train"] & w["deployed_capital"].notna()][
        ["wallet", EXPOSURE_PNL_COL, "deployed_capital"]
    ]
    capture = wallet_capture_thresholds(fit, EXPOSURE_FRACS, EXPOSURE_PNL_COL, "deployed_capital")
    carve_threshold = {}
    for frac in CARVE_FRACS:
        rows_ = capture[capture["frac"] == frac]
        if not rows_.empty:
            carve_threshold[frac] = float(rows_["threshold"].iloc[0])
    if len(carve_threshold) < len(CARVE_FRACS):
        print(f"{wallet}: no positive train {EXPOSURE_PNL_COL} — some/all carve variants skipped")

    fig = make_subplots(
        rows=2,
        cols=1,
        subplot_titles=("Train", "Test"),
        specs=[[{"secondary_y": True}], [{"secondary_y": True}]],
    )

    for row, (period, mask) in enumerate(
        [("train", w["is_train"]), ("test", ~w["is_train"])], start=1
    ):
        sub = w[mask].sort_values("dt")
        if sub.empty:
            continue

        fig.add_trace(
            go.Scatter(
                x=sub["dt"],
                y=sub["pnl"].cumsum(),
                mode="lines",
                name="pnl",
                legendgroup="pnl",
                showlegend=(row == 1),
                line=dict(color=LINE_COLORS["pnl"], width=2),
            ),
            row=row, col=1, secondary_y=False,
        )
        for c in COPYABLE_PNL_COLS:
            fig.add_trace(
                go.Scatter(
                    x=sub["dt"],
                    y=sub[c].cumsum(),
                    mode="lines",
                    name=c,
                    legendgroup=c,
                    showlegend=(row == 1),
                    line=dict(color=LINE_COLORS[c], width=2, dash="dash"),
                ),
                row=row, col=1, secondary_y=False,
            )
            exp = copyable_exposure_series(sub, f"{c}_exposure")
            if not exp.empty:
                fig.add_trace(
                    go.Scatter(
                        x=exp["dt"],
                        y=exp["exposure"],
                        mode="lines",
                        name=f"{c} exposure",
                        legendgroup=f"{c}_exp",
                        showlegend=(row == 1),
                        line=dict(color=LINE_COLORS[c], width=1.5, dash="dot"),
                        opacity=0.75,
                    ),
                    row=row, col=1, secondary_y=True,
                )

        # carved copyable PnL + exposure per bankroll frac (train-fit T_{w,f})
        for frac in CARVE_FRACS:
            if frac not in carve_threshold:
                continue
            carved = sub[
                sub["deployed_capital"].notna()
                & (sub["deployed_capital"] >= carve_threshold[frac])
            ]
            if carved.empty:
                continue
            color = FRAC_COLORS[frac]
            for c in COPYABLE_PNL_COLS:
                group = f"{c} (frac {frac:g})"
                fig.add_trace(
                    go.Scatter(
                        x=carved["dt"],
                        y=carved[c].cumsum(),
                        mode="lines",
                        name=group,
                        legendgroup=f"{c}_frac{frac:g}",
                        showlegend=(row == 1),
                        line=dict(color=color, width=1.5, dash="dash"),
                        opacity=0.9,
                    ),
                    row=row, col=1, secondary_y=False,
                )
                exp = copyable_exposure_series(carved, f"{c}_exposure")
                if not exp.empty:
                    fig.add_trace(
                        go.Scatter(
                            x=exp["dt"],
                            y=exp["exposure"],
                            mode="lines",
                            name=f"{c} exposure (frac {frac:g})",
                            legendgroup=f"{c}_exp_frac{frac:g}",
                            showlegend=(row == 1),
                            line=dict(color=color, width=1.5, dash="dot"),
                            opacity=0.7,
                        ),
                        row=row, col=1, secondary_y=True,
                    )


    fig.update_layout(
        title=f"Wallet {wallet} — cumulative PnL, copyable PnL (variants + carve fracs) and exposure",
        template="plotly_white",
        height=800,
    )
    fig.update_yaxes(title_text="Cumulative PnL (USDC)", secondary_y=False)
    fig.update_yaxes(title_text="Exposure (USDC)", secondary_y=True)
    fig.show(renderer="browser")


In [34]:
df_full['is_copyable'] = df_full['avail_copy_count_5m_100'] >=1

In [35]:
df_full[df_full['wallet'].isin(wallet_cohorts['copyable_group']['wallet'])] \
    .groupby('wallet') \
    .agg(
        trade_count=('wallet', 'count'),
        copyable_count=('is_copyable', 'sum')
    ) \
    .assign(copyable_ratio=lambda x: x['copyable_count'] / x['trade_count']) \
    .reset_index().sort_values('trade_count', ascending=False).head(5)

,wallet,trade_count,copyable_count,copyable_ratio
3,0x1f7105a18d9f36aef7c83e5df210e57cf487b2e4,677,228,0.34
12,0x6b23c73159b753c96d3c1ed94413f113300c04a3,615,582,0.95
21,0xb8d757413fe4d4ec4716390938f12100614f5ac7,356,30,0.08
17,0x93118ca14040b05c7a7d7c71cd4c2d6304a67c73,195,103,0.53
9,0x46ca395d519ecc8fa9a4fa5c77917f762cf8ac4c,173,132,0.76


In [36]:
if(WALLET_FILTER is not None):
    wallet_cohorts['copyable_group'] = wallet_cohorts['copyable_group'].query("wallet in @WALLET_FILTER").copy().reset_index(drop=True)

In [37]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)


In [38]:
filter_wallets = set(wallet_cohorts['predicting_group']["wallet"])
df_test =  df_full[
    (df_full['wallet'].isin(filter_wallets))
    ].groupby(['question', 'condition_id', 'outcome']).agg(
    # ].groupby(['wallet']).agg(
        pnl_sum=('pnl', 'sum'),
        trade_count=('pnl', 'size'),
        wallet_count=('wallet', 'nunique'),
    ).reset_index().sort_values('pnl_sum', key=abs, ascending=False)

big_conditions = df_test.groupby('condition_id').agg(
    abs_pnl_sum=('pnl_sum', lambda x: abs(x).sum()),
    min_abs_pnl_pct=('pnl_sum', lambda x: abs(x).min() / abs(x).sum() if abs(x).sum() > 0 else 0)
).reset_index().sort_values('abs_pnl_sum', ascending=False)



print(df_test['pnl_sum'].sum())
df_test = df_test[df_test['condition_id'].isin(big_conditions['condition_id'])].sort_values('question', ascending=False).merge(big_conditions, on='condition_id', how='left')

print(df_test[df_test['min_abs_pnl_pct']<0.1]['pnl_sum'].sum()) 

df_test[df_test['min_abs_pnl_pct'] < 0.1].sort_values(['abs_pnl_sum'], ascending=False).head(2)

248405.05074672468
-6171.3606684900005


,question,condition_id,outcome,pnl_sum,trade_count,wallet_count,abs_pnl_sum,min_abs_pnl_pct
1781,LoL: Cloud9 vs LYON - Game 2 Winner,0x5421cbfe4b57840d24063a9637c5e5b0c69590673d20...,Cloud9,-13.09,2,1,4697.47,0.00
1780,LoL: Cloud9 vs LYON - Game 2 Winner,0x5421cbfe4b57840d24063a9637c5e5b0c69590673d20...,LYON,4684.38,4,1,4697.47,0.00


In [39]:
wallets = set(wallet_cohorts['copyable_group']['wallet'])

df = df_full #[~df_full["is_train"]].copy()
print(len(df))

df_wallets = df[
    (df['wallet'].isin(wallets))
    & ~df['is_train']
    ].copy()
print(len(df_wallets))

df = df_wallets.groupby('condition_id').agg(
    pnl=('pnl', 'sum'),
    notional=('notional', 'sum'),
    sell_count=('side', lambda x: (x == 'SELL').sum()),
    buy_count=('side', lambda x: (x == 'BUY').sum()),
    wallets=('wallet', lambda x: x.nunique()),
).sort_values(by="pnl", key=np.abs, ascending=False).merge(mdf, on="condition_id", how="left")

2643409
1035


In [40]:
wallet_set = set(wallet_cohorts['copyable_group']['wallet'])
print(f"Copyable wallets: {len(wallet_set)} with fills {df_wallets[df_wallets['wallet'].isin(wallet_set)].shape[0]} with total PnL: {df_wallets[df_wallets['wallet'].isin(wallet_set)]['pnl'].sum():,.2f}")
df_wallets[df_wallets['wallet'].isin(wallet_set)]['trade_pnl'].sum()

Copyable wallets: 26 with fills 1035 with total PnL: 140,947.37


np.float64(140947.37028811002)

In [41]:
# ── Copyable-group test stats: wallet vs copyable variants ──────────────────
# Cohort-level stats on the test period for the copyable_group: the wallets'
# own PnL (trade_pnl) and each copyable PnL variant. ROI uses the capital
# deployed (sum of per-fill exposure). Sharpe is the annualized daily-PnL
# Sharpe, zero-filled across the test date range. Exposure quantiles come
# from the capital-at-risk time profile (fills add exposure, released 1d
# after the contract's last trade).
#
# In addition to the full rows, each variant also gets a row per exposure
# carve (`EXPOSURE_FRACS`, frac < 1). The carve is a bankroll-discipline
# filter: per wallet keep the fills whose deployed capital
# (position * price, the post-trade USDC value of the book on that
# condition/outcome) is >= T_{w,f}, where T is fit on TRAIN via
# wallet_capture_thresholds (the smallest exposure whose retained fills
# capture fraction `f` of the wallet's train copyable PnL; f == 1 keeps
# everything). The same train-fit thresholds are applied to the test fills;
# wallets with no positive train PnL get no threshold and are kept whole.

from polymarket_analysis.visualization.wallet_plots import copyable_exposure_series
from polymarket_analysis.wallet_selection.exposure import wallet_capture_thresholds

df_full["deployed_capital"] = df_full["position"] * df_full["price"]

_copyable_wallets = wallet_cohorts["copyable_group"]["wallet"]
test_fills = df_full[
    (~df_full["is_train"]) & (df_full["wallet"].isin(_copyable_wallets))
].copy()
train_fills = df_full[
    (df_full["is_train"]) & (df_full["wallet"].isin(_copyable_wallets))
].copy()

PNL_TO_QTY = {
    COPYABLE_PNL_BASE: "copyable_qty_5m_100",
    **{f"copyable_pnl_{k}": cfg["qty_col"] for k, cfg in COPYABLE_VARIANTS.items()},
}

for key, qty_col in PNL_TO_QTY.items():
    test_fills[f"{key}_exposure"] = test_fills["price"] * test_fills[qty_col]


_capture = wallet_capture_thresholds(
    train_fills, EXPOSURE_FRACS, EXPOSURE_PNL_COL, "deployed_capital"
)


def _annualized_daily_sharpe(daily: pd.Series, periods_per_year: float = 365.0) -> float:
    if len(daily.dropna()) < 2:
        return float("nan")
    full = pd.date_range(daily.index.min(), daily.index.max(), freq="D")
    s = daily.reindex(full, fill_value=0.0)
    sd = s.std()
    if sd == 0:
        return float("nan")
    return float(s.mean() / sd * np.sqrt(periods_per_year))


EXPOSURE_QUANTILES = [0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
CARVE_FRACS = sorted({f for f in EXPOSURE_FRACS if f < 1}, reverse=True)

variant_specs = [("wallet", "trade_pnl", "notional")]
# The size overlay is fitted for the predicting group; exclude it from the
# copyable-group stats so the two cohorts are not mixed.
variant_specs += [
    (c, c, f"{c}_exposure")
    for c in COPYABLE_PNL_COLS
    if c != "copyable_pnl_size"
]


def _stats_row(sub: pd.DataFrame, label: str, pnl_col: str, exp_col: str, n_total: int) -> dict:
    total_pnl = sub[pnl_col].sum()
    total_capital = sub[exp_col].sum()
    daily = sub.assign(_d=sub["dt"].dt.floor("D")).groupby("_d")[pnl_col].sum()
    profile = copyable_exposure_series(sub, exp_col, bucket_freq="D")
    exp_qs = {
        f"exp_p{int(q * 100)}": (
            profile["exposure"].quantile(q) if not profile.empty else float("nan")
        )
        for q in EXPOSURE_QUANTILES
    }
    return {
        "variant": label,
        "total_pnl": total_pnl,
        "total_roi": (
            total_pnl / total_capital
            if pd.notna(total_capital) and total_capital > 0
            else float("nan")
        ),
        "sharpe": _annualized_daily_sharpe(daily),
        "kept_rows": len(sub),
        "kept_rows_pct": len(sub) / n_total if n_total else float("nan"),
        **exp_qs,
        "exp_max": profile["exposure"].max() if not profile.empty else float("nan"),
    }


n_total = len(test_fills)
stat_rows = []
for label, pnl_col, exp_col in variant_specs:
    stat_rows.append(_stats_row(test_fills, label, pnl_col, exp_col, n_total))
    for frac in CARVE_FRACS:
        T = test_fills["wallet"].map(
            _capture[_capture["frac"] == frac].set_index("wallet")["threshold"]
        ).fillna(0.0)
        carved = test_fills[
            test_fills["deployed_capital"].notna() & (test_fills["deployed_capital"] >= T)
        ]
        if carved.empty:
            continue
        stat_rows.append(_stats_row(carved, f"{label} (frac {frac:g})", pnl_col, exp_col, n_total))

copyable_group_test_stats = pd.DataFrame(stat_rows)
copyable_group_test_stats


,variant,total_pnl,total_roi,sharpe,kept_rows,kept_rows_pct,exp_p25,exp_p50,exp_p75,exp_p90,exp_p95,exp_p99,exp_max
0,wallet,140947.37,0.08,5.43,1035,1.00,8372.83,23127.14,40112.66,64890.47,100149.26,123508.92,149472.17
1,copyable_pnl,78085.37,0.07,3.17,1035,1.00,3068.04,7559.07,30465.90,57019.40,64737.90,79370.42,86767.33
2,copyable_pnl_flow,-5148.48,-0.33,-1.13,1035,1.00,0.00,0.00,0.00,19.76,498.39,6458.13,10000.00
3,copyable_pnl_flow_strict,-5297.19,-0.37,-1.17,1035,1.00,0.00,0.00,0.00,0.00,9.88,6458.13,10000.00
4,copyable_pnl_price_size,1456.55,0.17,1.49,1035,1.00,0.00,0.00,0.00,314.60,1333.60,2148.19,2891.00


In [42]:
MARKET = '0xaabf22a03217285585592f3082f27e2b0f4c1cee2bb4003e84817100bbcf1938'

market_def = mdf.loc[mdf['condition_id'] == MARKET].head(1).squeeze()
print(f"Market: {market_def}")

dfc = df_full[(df_full['condition_id'] == MARKET)
              & (df_full['wallet'].isin(wallets))].copy()
# dfc['bucket'] = dfc['dt'].dt.floor('1h')


dfc = dfc.groupby(['dt', 'wallet', 'side', 'outcome']).agg( 
        pnl=('pnl', 'sum'),
        notional=('notional', 'sum'),
        quantity=('quantity', 'sum'),
        position=('position', 'last'),
        avg_price=('price', lambda x: (x * dfc.loc[x.index, 'quantity']).sum() / dfc.loc[x.index, 'quantity'].sum()),
        copyable_qty_5m_100=('copyable_qty_5m_100', 'sum'),
        copyable_pnl=('copyable_pnl', 'sum'),
)[['pnl', 'position', 'notional', 'quantity', 'avg_price', 'copyable_qty_5m_100', 'copyable_pnl']].reset_index().sort_values(['dt', 'wallet', 'side', 'outcome'])

Market: Empty DataFrame
Columns: [condition_id, end_date_iso, question, tags, primary_tag, winner_token_id, outcome]
Index: []


/var/folders/j8/0dbnwk8n6m933m843h7hb88w0000gn/T/ipykernel_45045/3456545487.py:11: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [43]:
# dominant tags
wallet_fills = df_full[df_full['wallet'].isin(wallet_cohorts['multi_filter']['wallet'])]

dominant_tags = (
    wallet_fills
    .groupby(['wallet', 'primary_tag'], as_index=False)[['quantity', 'pnl']].sum()
    .sort_values(['wallet', 'quantity'], ascending=[True, False])
    .assign(total_qty=lambda df: df.groupby('wallet')['quantity'].transform('sum'))
    .drop_duplicates('wallet')
    .assign(primary_tag_ratio=lambda df: df['quantity'] / df['total_qty'])
    .rename(columns={
        'quantity': 'primary_tag_qty'
    })[['wallet', 'primary_tag', 'primary_tag_qty', 'primary_tag_ratio', 'pnl']]
)

In [44]:
watched_wallets = wallet_cohorts['multi_filter'].sort_values("copyable_pnl", key=abs, ascending=False)['wallet'].head(10).to_list()
for w in watched_wallets:
    wallet_df = wallet_vol_train[wallet_vol_train['wallet'] == w]
    wallet_cohorts[w] = wallet_df.copy().reset_index(drop=True)

In [45]:
import plotly.graph_objects as go

from polymarket_analysis.visualization.wallet_plots import copyable_exposure_series

dp = wallet_cohorts['predicting_group']
dp = dp[dp['wallet'] != '0x1f7105a18d9f36aef7c83e5df210e57cf487b2e4']
wallet_cohorts['predicting_group'] = dp
print(f"copyable_group wallets: {len(wallet_cohorts['copyable_group']):,}")
print(f"predicting_group wallets: {len(wallet_cohorts['predicting_group']):,}")

pd.set_option('display.max_rows', 100)

PNL_TO_QTY = {
    COPYABLE_PNL_BASE: "copyable_qty_5m_100",
    **{f"copyable_pnl_{k}": cfg["qty_col"] for k, cfg in COPYABLE_VARIANTS.items()},
}

plot_rows = []
exposure_rows = []

# The size overlay is fitted for the predicting group; keep it out of the
# copyable-group traces so the two cohorts are not mixed.
_group_variant_cols = {
    'copyable': [c for c in COPYABLE_PNL_COLS if c != 'copyable_pnl_size'],
    'predicting': list(COPYABLE_PNL_COLS),
}

for group_label, cohort_name in [('copyable', 'copyable_group'), ('predicting', 'predicting_group')]:
    wallets = set(wallet_cohorts[cohort_name]['wallet'])
    group_trades = df_full[
        (df_full['wallet'].isin(wallets))
        & (~df_full['is_train'])
    ][['dt', 'side', 'price', 'pnl', 'last_condition_trade_ts', 'wallet',
       'notional', 'deployed_capital',
       *COPYABLE_PNL_COLS, *PNL_TO_QTY.values()]].copy()

    # per-fill copy exposure (price × copyable qty, BUY only)
    for pnl_col, qty_col in PNL_TO_QTY.items():
        group_trades[f'{pnl_col}_exposure'] = np.where(
            group_trades[qty_col] > 0,
            group_trades['price'] * group_trades[qty_col],
            np.nan,
        )

    # Copy flow-limited variant exposure columns from df_full (created in the
    # flow-filter section) so the plots below can use them.
    for _c in COPYABLE_PNL_COLS:
        _e = f'{_c}_exposure'
        if _e in df_full.columns and _e not in group_trades.columns:
            group_trades[_e] = df_full.loc[group_trades.index, _e].values

    _variant_cols = _group_variant_cols[group_label]
    daily = (
        group_trades
        .assign(plot_dt=group_trades['dt'].dt.floor('D'))
        .groupby('plot_dt', as_index=False)[['pnl', *_variant_cols]]
        .sum()
        .sort_values('plot_dt')
        .reset_index(drop=True)
    )
    daily['cum_trade_pnl'] = daily['pnl'].cumsum()
    for c in _variant_cols:
        daily[f'cum_{c}'] = daily[c].cumsum()
    daily['wallet_group'] = group_label
    plot_rows.append(daily)

    for c in _variant_cols:
        exp = copyable_exposure_series(group_trades, f'{c}_exposure', bucket_freq='D')
        exp['wallet_group'] = group_label
        exp['variant'] = c
        exposure_rows.append(exp)

group_perf = pd.concat(plot_rows, ignore_index=True)
group_exposure = pd.concat(exposure_rows, ignore_index=True)

fig = go.Figure()
colors = {'copyable': '#1f77b4', 'predicting': '#ff7f0e'}
dashes = ['dash', 'dot', 'dashdot']
for group_label in ['copyable', 'predicting']:
    g = group_perf[group_perf['wallet_group'] == group_label]
    fig.add_trace(
        go.Scatter(
            x=g['plot_dt'],
            y=g['cum_trade_pnl'],
            mode='lines',
            name=f'{group_label} group trade pnl',
            line=dict(color=colors[group_label], width=2),
        )
    )
    for i, c in enumerate(_group_variant_cols[group_label]):
        fig.add_trace(
            go.Scatter(
                x=g['plot_dt'],
                y=g[f'cum_{c}'],
                mode='lines',
                name=f'{group_label} {c}',
                line=dict(color=colors[group_label], width=2, dash=dashes[i % len(dashes)]),
            )
        )
    for c in _group_variant_cols[group_label]:
        e = group_exposure[
            (group_exposure['wallet_group'] == group_label)
            & (group_exposure['variant'] == c)
        ]
        fig.add_trace(
            go.Scatter(
                x=e['dt'],
                y=e['exposure'],
                mode='lines',
                name=f'{group_label} {c} exposure',
                line=dict(color=colors[group_label], width=1.5, dash='dot'),
                opacity=0.75,
                yaxis='y2',
            )
        )

fig.update_layout(
    title='Stage1 wallet-group performance on test period — copyable PnL and exposure on right axis',
    xaxis_title='Date',
    yaxis_title='Cumulative PnL (USDC)',
    yaxis2=dict(title='Exposure (USDC)', overlaying='y', side='right', showgrid=False),
    template='plotly_white',
)
fig.show()


copyable_group wallets: 26
predicting_group wallets: 12


In [46]:
copyable_wallets = wallet_cohorts['copyable_group'].copy().assign(wallet_group='copyable')
predicting_wallets = wallet_cohorts['predicting_group'].copy().assign(wallet_group='predicting')

selected_wallets = (
    pd.concat([copyable_wallets, predicting_wallets], ignore_index=True)
    .drop_duplicates(subset=['wallet', 'wallet_group'])
    .reset_index(drop=True)
)

selected_wallets.to_parquet(WORKSPACE_DIR / 'selected_wallets_v2.parquet', index=False)
print(f"Saved selected wallets: {len(selected_wallets):,}")
print(selected_wallets['wallet_group'].value_counts().to_dict())


Saved selected wallets: 38
{'copyable': 26, 'predicting': 12}


In [47]:
len(df_wallets[df_wallets['is_train'] == False])

1035

## Wallet PnL over time

Three independent plots:
- **Training plot** — cohort aggregate cumulative PnL over the training period only
- **Test plot** — cohort aggregate cumulative PnL over the test period only (starts from 0)
- **Individual plot** — per-wallet cumulative PnL spanning train + test, with a train/test split
  vline and wallet address labels; test portion resets to 0 at the split boundary

Set `PLOT_WALLET_PNL = False` to skip (e.g. when running headlessly).


In [48]:
PLOT_WALLET_PNL  = True
TOP_N_INDIVIDUAL = 10   # wallets shown in panel 1 per cohort


In [49]:
df_fills = df_full.copy()
df_fills["end_date_iso"] = pd.to_datetime(df_fills["last_condition_trade_ts"], utc=True).dt.floor("D")

# per-variant fill exposure: capital needed to fill the copyable qty at the fill price
PNL_TO_QTY = {
    COPYABLE_PNL_BASE: "copyable_qty_5m_100",
    **{f"copyable_pnl_{k}": cfg["qty_col"] for k, cfg in COPYABLE_VARIANTS.items()},
}
for pnl_col, qty_col in PNL_TO_QTY.items():
    df_fills[f"{pnl_col}_exposure"] = np.where(
        df_fills[qty_col] > 0,
        df_fills["price"] * df_fills[qty_col],
        np.nan,
    )

# Normalise grouped schema → canonical fill-level names
if "total_quantity" in df_fills.columns and "quantity" not in df_fills.columns:
    df_fills = df_fills.rename(columns={
        "total_quantity": "quantity",
        "avg_price": "price",
        "trade_value_usdc": "usdc_amount",
    })

df_fills["usdc_amount"]      = df_fills["usdc_amount"].astype(float)
df_fills["final_value_usdc"] = df_fills["final_value_usdc"].astype(float)
df_fills["quantity"]         = df_fills["quantity"].astype(float)


In [50]:
filter_fills = df_fills[
    (df_fills['wallet'].isin(wallet_cohorts['multi_filter']['wallet']))
    & (df_fills['side'] == 'BUY')
    # & (TAGS is None or df_fills['primary_tag'].isin(TAGS))
    & (df_fills["is_train"] == False)
    ].copy().reset_index(drop=True)

print(len(filter_fills))
# filter_fills = filter_fills[filter_fills['copyable_qty_5m_100'] >= 1].copy().reset_index(drop=True)
print(len(filter_fills))
filter_fills.head(2)

1035
1035


,wallet,condition_id,token_id,dt,side,position,quantity,price,usdc_amount,final_value_usdc,trade_pnl,copyable_pnl,token_winner,final_price,last_condition_trade_ts,tx_hash,num_fills,is_train,copyable_qty_5m_100,avail_copy_qty_5m_100,avail_copy_total_vol_5m_100,avail_copy_count_5m_100,flow_own_usdc_60,usdc_all_60,price_chg_last_own,end_date_iso,question,tags,primary_tag,winner_token_id,outcome,pnl,notional,size_share,deployed_capital,is_carved,flow_usdc_share_60,copyable_qty_2x,copyable_pnl_2x,copyable_qty_0.5x,copyable_pnl_0.5x,copyable_notional,roi,copyable_roi,bucket,flow_pass_nonstrict,flow_pass_strict,copyable_pnl_flow,copyable_notional_flow,copyable_pnl_flow_exposure,copyable_pnl_flow_strict,copyable_notional_flow_strict,copyable_pnl_flow_strict_exposure,size_pass,copyable_pnl_size,copyable_notional_size,copyable_pnl_size_exposure,price_size_pass,copyable_pnl_price_size,copyable_notional_price_size,copyable_pnl_price_size_exposure,is_copyable,copyable_pnl_exposure
0,0x46ca395d519ecc8fa9a4fa5c77917f762cf8ac4c,0x02226c4cc97b927409bb64e52bd183a86a1270be6b87...,6557202114551230021603877159126676091888256419...,2026-09-02 17:43:22+00:00,BUY,2799.84,1136.36,0.88,1000.00,1136.36,136.36,70.59,True,1.00,2026-09-02 19:54:09+00:00,0xf19c29766169894ff3de17d09bb0b6e349e4434d4674...,1,False,588.23,588.23,511.76,1.00,478.09,2061.66,0.00,2026-09-02 00:00:00+00:00,Will Udinese Calcio win on 2026-09-02?,"[Sports, Soccer, Games, Coppa Italia]",Sports,6557202114551230021603877159126676091888256419...,Yes,136.36,1000.00,0.49,2463.86,True,0.23,304.49,36.54,294.11,35.29,517.64,0.14,0.14,2026-09-02 17:45:00+00:00,False,False,0.00,0.00,0.00,0.00,0.00,0.00,False,0.00,0.00,0.00,False,0.00,0.00,0.00,True,517.64
1,0x58ce8cec3cbc6eb3dbbe8cd39436bc0dd2ebe206,0x04cfc9a4750296502a38b7dd244e7d7c7e277576f62a...,1760222650468434636851050106047812288879138534...,2026-09-02 11:41:01+00:00,BUY,6000.00,3000.00,0.53,1590.00,3000.00,1410.00,35.28,True,1.00,2026-09-02 13:17:30+00:00,0xca03143ce0d3b27bf50a7af668722bfa2bfd54dbf3d8...,1,False,75.05,75.05,36.03,3.00,5677.34,109165.56,0.01,2026-09-02 00:00:00+00:00,LoL: Hanwha Life Esports vs T1 (BO5) - LCK Pla...,"[Sports, Esports, league of legends, Games]",Sports,1760222650468434636851050106047812288879138534...,Hanwha Life Esports,1410.00,1590.00,0.01,3180.00,True,0.05,1.88,0.88,1.88,0.88,39.78,0.89,0.89,2026-09-02 11:40:00+00:00,False,False,0.00,0.00,0.00,0.00,0.00,0.00,True,0.00,0.00,0.00,True,35.28,39.78,39.78,True,39.78


In [51]:
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.max_rows', 1000)

In [52]:
filter_fills["bucket"] = filter_fills["dt"].dt.floor("5min")

MAX_EXPOSURE = 100000

df = filter_fills.copy()

# The size overlay is fitted for the predicting group; keep it out of the
# multi_filter (copyable-group) stream.
_variant_cols = [c for c in COPYABLE_PNL_COLS if c != "copyable_pnl_size"]

# scale each copyable variant to the per-fill MAX_EXPOSURE capital budget
for c in _variant_cols:
    exp_col = f"{c}_exposure"
    scale = np.minimum(1, MAX_EXPOSURE / df[exp_col].replace(0, np.nan))
    df[f"scaled_{c}"] = df[c] * scale
    df[f"scaled_{exp_col}"] = df[exp_col] * scale

df.head(1)


,wallet,condition_id,token_id,dt,side,position,quantity,price,usdc_amount,final_value_usdc,trade_pnl,copyable_pnl,token_winner,final_price,last_condition_trade_ts,tx_hash,num_fills,is_train,copyable_qty_5m_100,avail_copy_qty_5m_100,avail_copy_total_vol_5m_100,avail_copy_count_5m_100,flow_own_usdc_60,usdc_all_60,price_chg_last_own,end_date_iso,question,tags,primary_tag,winner_token_id,outcome,pnl,notional,size_share,deployed_capital,is_carved,flow_usdc_share_60,copyable_qty_2x,copyable_pnl_2x,copyable_qty_0.5x,copyable_pnl_0.5x,copyable_notional,roi,copyable_roi,bucket,flow_pass_nonstrict,flow_pass_strict,copyable_pnl_flow,copyable_notional_flow,copyable_pnl_flow_exposure,copyable_pnl_flow_strict,copyable_notional_flow_strict,copyable_pnl_flow_strict_exposure,size_pass,copyable_pnl_size,copyable_notional_size,copyable_pnl_size_exposure,price_size_pass,copyable_pnl_price_size,copyable_notional_price_size,copyable_pnl_price_size_exposure,is_copyable,copyable_pnl_exposure,scaled_copyable_pnl,scaled_copyable_pnl_exposure,scaled_copyable_pnl_flow,scaled_copyable_pnl_flow_exposure,scaled_copyable_pnl_flow_strict,scaled_copyable_pnl_flow_strict_exposure,scaled_copyable_pnl_price_size,scaled_copyable_pnl_price_size_exposure
0,0x46ca395d519ecc8fa9a4fa5c77917f762cf8ac4c,0x02226c4cc97b927409bb64e52bd183a86a1270be6b87e6be74a0c3404527ad1d,65572021145512300216038771591266760918882564194904924275607611590622107741124,2026-09-02 17:43:22+00:00,BUY,2799.84,1136.36,0.88,1000.00,1136.36,136.36,70.59,True,1.00,2026-09-02 19:54:09+00:00,0xf19c29766169894ff3de17d09bb0b6e349e4434d467466967c28d68f7fbff877,1,False,588.23,588.23,511.76,1.00,478.09,2061.66,0.00,2026-09-02 00:00:00+00:00,Will Udinese Calcio win on 2026-09-02?,"[Sports, Soccer, Games, Coppa Italia]",Sports,65572021145512300216038771591266760918882564194904924275607611590622107741124,Yes,136.36,1000.00,0.49,2463.86,True,0.23,304.49,36.54,294.11,35.29,517.64,0.14,0.14,2026-09-02 17:40:00+00:00,False,False,0.00,0.00,0.00,0.00,0.00,0.00,False,0.00,0.00,0.00,False,0.00,0.00,0.00,True,517.64,70.59,517.64,0.00,0.00,0.00,0.00,0.00,0.00


In [53]:
# ── exposure + PnL over time, per copyable variant ────────────────────────────
agg_dict = {"wallet_pnl": ("trade_pnl", "sum")}
agg_dict.update({f"scaled_{c}": (f"scaled_{c}", "sum") for c in _variant_cols})
agg_dict.update({f"scaled_{c}_exposure": (f"scaled_{c}_exposure", "sum") for c in _variant_cols})
df_plot = df.groupby("end_date_iso").agg(**agg_dict).reset_index()

df.sort_values("bucket", inplace=True)
df["wallet_pnl_cumsum"] = df["trade_pnl"].cumsum()
for c in _variant_cols:
    df[f"scaled_{c}_cumsum"] = df[f"scaled_{c}"].cumsum()


def exposure_timeline(sub, c, exp_col=None):
    """Cumulative exposure over time for a subset frame / copyable variant.

    Fills add exposure at buy time; positions resolve at EOD of end_date_iso
    (+1 day), releasing the exposure (negative delta)."""
    if exp_col is None:
        exp_col = f"scaled_{c}_exposure"
    daily = sub.groupby("end_date_iso")[exp_col].sum().reset_index()
    resolution = daily.rename(columns={exp_col: "exposure_delta"})
    resolution["dt"] = pd.to_datetime(resolution["end_date_iso"]) + pd.Timedelta(days=1)
    resolution["exposure_delta"] = -resolution["exposure_delta"]
    resolution["variant"] = c
    buy = sub[["bucket", exp_col]].rename(columns={"bucket": "dt", exp_col: "exposure_delta"})
    buy["variant"] = c
    ev = pd.concat(
        [resolution[["dt", "variant", "exposure_delta"]], buy[["dt", "variant", "exposure_delta"]]],
        ignore_index=True,
    )
    ev.sort_values("dt", inplace=True)
    ev["exposure"] = ev.groupby("variant")["exposure_delta"].cumsum()
    return ev


variant_exposure = pd.concat(
    [exposure_timeline(df, c) for c in _variant_cols], ignore_index=True)

import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df["bucket"],
    y=df["wallet_pnl_cumsum"],
    mode="lines",
    name="wallet_pnl",
))

for c in _variant_cols:
    fig.add_trace(go.Scatter(
        x=df["bucket"],
        y=df[f"scaled_{c}_cumsum"],
        mode="lines",
        name=f"scaled_{c}",
        line=dict(dash="dash"),
    ))
for c in _variant_cols:
    exp_df = variant_exposure[variant_exposure["variant"] == c]
    fig.add_trace(go.Scatter(
        x=exp_df["dt"],
        y=exp_df["exposure"],
        mode="lines",
        name=f"exposure ({c})",
        line=dict(dash="dot"),
    ))
# ── deployed-capital carved copies ───────────────────────────────────────────
# Bankroll-discipline carve: per wallet keep the fills whose deployed capital
# (position * price, post-trade USDC book value on that condition / outcome)
# is >= T_{w,f}, with T fit on TRAIN via wallet_capture_thresholds (the
# smallest exposure capturing fraction `f` of the wallet's train copyable PnL;
# f == 1 keeps everything). The same train-fit T is applied to the test frame
# `df`; PnL / exposure lines are rebuilt on the carved subset.
# frac == 1 is the uncarved copy_all baseline (already drawn above).
import plotly.colors as _pl_colors

df_fills["deployed_capital"] = df_fills["position"] * df_fills["price"]
df["deployed_capital"] = df["position"] * df["price"]

_fit = df_fills[
    df_fills["wallet"].isin(wallet_cohorts["multi_filter"]["wallet"])
    & (df_fills["side"] == "BUY")
    & ((TAGS is None) or df_fills["primary_tag"].isin(TAGS))
    & df_fills["is_train"]
    & df_fills["deployed_capital"].notna()
][["wallet", EXPOSURE_PNL_COL, "deployed_capital"]].copy()

_capture = wallet_capture_thresholds(
    _fit, EXPOSURE_FRACS, EXPOSURE_PNL_COL, "deployed_capital"
)

_exposed_fracs = sorted({f for f in EXPOSURE_FRACS if f < 1}, reverse=True)
FRAC_COLOR = {f: _pl_colors.qualitative.Plotly[i % 10] for i, f in enumerate(_exposed_fracs)}

carve_sizes = {}   # frac -> kept rows (shared across all copyable variants)
carve_index = {}   # frac -> kept index

for frac in _exposed_fracs:
    T = df["wallet"].map(
        _capture[_capture["frac"] == frac].set_index("wallet")["threshold"]
    ).fillna(0.0)
    carved = df[
        df["deployed_capital"].notna() & (df["deployed_capital"] >= T)
    ].sort_values("bucket")
    if carved.empty:
        continue
    color = FRAC_COLOR[frac]
    for c in _variant_cols:
        fig.add_trace(go.Scatter(
            x=carved["bucket"],
            y=carved[f"scaled_{c}"].cumsum(),
            mode="lines",
            name=f"scaled_{c} (frac {frac:g})",
            line=dict(color=color, dash="dash", width=2.0),
            opacity=0.95,
        ))
        expos = exposure_timeline(carved, c)
        fig.add_trace(go.Scatter(
            x=expos["dt"],
            y=expos["exposure"],
            mode="lines",
            name=f"exposure ({c}) frac {frac:g}",
            line=dict(color=color, dash="dot", width=1.5),
            opacity=0.75,
        ))
    carve_sizes[frac] = len(carved)
    carve_index[frac] = carved.index

fig.show(renderer="browser")

# ── validation: how much does each carve actually keep? ─────────────────────
# With train-fit deployed-capital thresholds the carve is shared across all
# copyable variants (same fill set per frac). Show the retained fill share,
# deployed-capital share, and base copyable PnL share on the test frame.
print(f"total test fills (df): {len(df):,}")
validation_rows = []
for frac in _exposed_fracs:
    idx = carve_index.get(frac)
    if idx is None:
        continue
    carved = df.loc[idx]
    validation_rows.append({
        "frac": frac,
        "kept_rows": len(carved),
        "kept_rows_pct": len(carved) / len(df),
        "kept_deployed_cap_pct": (
            carved["deployed_capital"].sum() / df["deployed_capital"].sum()
            if df["deployed_capital"].sum() else float("nan")
        ),
        "kept_copyable_pnl_pct": (
            carved[COPYABLE_PNL_BASE].sum() / df[COPYABLE_PNL_BASE].sum()
            if df[COPYABLE_PNL_BASE].sum() else float("nan")
        ),
    })
pd.DataFrame(validation_rows).round(3)


total test fills (df): 1,035


""


In [54]:
df_fills[(
    (df_fills['is_train'] == False)
    & (df_fills['wallet'].isin(wallet_set))
    & (df_fills['side'] == 'BUY')
          )].groupby(['primary_tag', 'side']).agg(
    copyable_pnl=('copyable_pnl', 'sum'),
    total_copyable_qty=('copyable_qty_5m_100', 'sum'),
    trade_pnl=('trade_pnl', 'sum'),
    trades=('condition_id', 'count'),
    condition_count=('condition_id', 'nunique'),
    wallets=('wallet', lambda x: x.nunique()),
).sort_values('copyable_pnl', key=abs, ascending=False).reset_index().head(10)

/var/folders/j8/0dbnwk8n6m933m843h7hb88w0000gn/T/ipykernel_45045/3940907672.py:5: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



,primary_tag,side,copyable_pnl,total_copyable_qty,trade_pnl,trades,condition_count,wallets
0,Sports,BUY,78085.37,1573110.79,140947.37,1035,211,26.00
1,Sports,SELL,0.00,0.00,0.00,0,0,NaN


In [55]:
from polymarket_analysis.visualization.wallet_plots import (
    plot_wallet_selection_pnl,
    plot_wallet_individual_pnl,
)

if PLOT_WALLET_PNL:
    # Plot the copyable sharpe terciles in place of the combined copyable_group
    plot_cohorts = {k: v for k, v in wallet_cohorts.items() if k != 'copyable_group'}

    # Column order per cohort: base copyable first, then the other variants.
    # The library draws per cohort: total → raw lines → carved fracs
    # (train-fit thresholds on the base copyable_pnl) → exposure lines last.
    COHORT_PNL_COLS = [COPYABLE_PNL_BASE]
    COHORT_PNL_COLS += [c for c in COPYABLE_PNL_COLS if c not in COHORT_PNL_COLS]
    assert len(COHORT_PNL_COLS) == len(set(COHORT_PNL_COLS))
    # Cohort aggregate PnL — training period
    fig_train = plot_wallet_selection_pnl(
        df_fills,
        plot_cohorts,
        split_date=TRAIN_END,
        period="train",
        title="Wallet selection cohorts — cohort cumulative PnL (training period, copyable PnL, exposure)",
        pnl_cols=COHORT_PNL_COLS,
        exposure_fracs=EXPOSURE_FRACS,
    )
    fig_train.show(renderer="browser")

    # Cohort aggregate PnL — test period (starts from 0)
    fig_test = plot_wallet_selection_pnl(
        df_fills,
        plot_cohorts,
        split_date=TRAIN_END,
        period="test",
        title="Wallet selection cohorts — cohort cumulative PnL (test period, copyable PnL, exposure)",
        pnl_cols=COHORT_PNL_COLS,
        exposure_fracs=EXPOSURE_FRACS,
    )
    fig_test.show(renderer="browser")

    # # Individual wallet lines (train + test) with split vline and labels
    # fig_ind = plot_wallet_individual_pnl(
    #     df_fills,
    #     wallet_cohorts,
    #     split_date=split_dt,
    #     top_n_individual=TOP_N_INDIVIDUAL,
    #     title="Individual wallet cumulative PnL (train + test, resets at split)",
    # )
    # fig_ind.show(renderer="browser")


## Flow filter backtest comparison

Compares three variants on test BUY fills of `multi_filter` wallets:
- **baseline**: no flow filter (current behavior)
- **nonstrict**: `price ∈ [0.20, 0.50]` AND `flow_usdc_share_60 ≤ −0.10`
- **strict**: nonstrict AND `price_chg_last_tok > 0`

In [56]:
# ── Flow filter backtest comparison ────────────────────────────────────────
# Load PnL columns for uncarved fills from processed data
_pnl_cols = ["wallet", "condition_id", "token_id", "tx_hash", "dt", "side",
             "trade_pnl", "copyable_pnl", "trade_value_usdc", "final_value_usdc"]
_pnl_ds = ds.dataset(TRADES_DIR, format="parquet")
_pnl_t = _pnl_ds.to_table(
    columns=_pnl_cols,
    filter=ds.field("wallet").isin(list(selected_wallets_flow)),
)
_pnl_df = _pnl_t.to_pandas()
_pnl_buy = _pnl_df[_pnl_df["side"] == "BUY"].copy()
_pnl_buy["notional"] = _pnl_buy["trade_value_usdc"].astype(float)
_pnl_buy["copyable_notional"] = _pnl_buy["notional"] * (
    _pnl_buy["copyable_pnl"] / _pnl_buy["trade_pnl"].replace(0, np.nan)
)
# Merge with uncarved flow fills on natural key
_merge_keys = ["wallet", "tx_hash", "condition_id", "token_id"]
for c in _merge_keys:
    _pnl_buy[c] = _pnl_buy[c].astype(str)
    flow_fills_uncarved[c] = flow_fills_uncarved[c].astype(str)
flow_fills_uncarved = flow_fills_uncarved.merge(
    _pnl_buy[_merge_keys + ["trade_pnl", "copyable_pnl", "notional", "copyable_notional"]],
    on=_merge_keys, how="left", suffixes=("", "_pnl"),
)
# Use the PnL columns from the merge (drop old ones if any)
for c in ["trade_pnl", "copyable_pnl", "notional", "copyable_notional"]:
    if f"{c}_pnl" in flow_fills_uncarved.columns:
        flow_fills_uncarved[c] = flow_fills_uncarved[f"{c}_pnl"]
        flow_fills_uncarved.drop(columns=[f"{c}_pnl"], inplace=True)
print(f"Uncarved fills with PnL: {flow_fills_uncarved['trade_pnl'].notna().sum():,} / {len(flow_fills_uncarved):,}")

def _agg_roi(g, pnl_col, not_col):
    _n = g[not_col].sum()
    return g[pnl_col].sum() / _n if _n > 0 else np.nan

# ── CARVED stream (df_full → filter_fills, the current pipeline) ───────────
_ff_wallets = set(wallet_cohorts['predicting_group']['wallet'])
_ff_base = df_fills[
    (df_fills['wallet'].isin(_ff_wallets))
    & (df_fills['side'] == 'BUY')
    & (df_fills['is_train'] == False)
].copy()

_variants_carved = {
    'baseline': np.ones(len(_ff_base), dtype=bool),
    'nonstrict': _ff_base['flow_pass_nonstrict'].values,
    'strict': _ff_base['flow_pass_strict'].values,
}

_rows = []
for _name, _mask in _variants_carved.items():
    _k = _ff_base[_mask]
    if len(_k) == 0:
        continue
    _rows.append({
        'stream': 'carved',
        'variant': _name,
        'fills': len(_k),
        'retention': len(_k) / len(_ff_base),
        'copyable_pnl': _k['copyable_pnl'].sum(),
        'copyable_roi': _agg_roi(_k, 'copyable_pnl', 'copyable_notional'),
        'actual_pnl': _k['trade_pnl'].sum(),
        'actual_roi': _agg_roi(_k, 'trade_pnl', 'notional'),
    })

# ── UNCARVED stream (flow_fills_uncarved, all BUY fills) ───────────────────
# PnL / notional are merged in above from the processed parquet; apply the same
# train/test split and flow masks so the two streams are comparable.
_unc_test = flow_fills_uncarved[flow_fills_uncarved['dt'] >= pd.Timestamp('2026-08-01', tz='UTC')].copy()
_unc_rows = []
for _name, _mask_col in [('baseline', None), ('nonstrict', 'flow_pass_nonstrict'), ('strict', 'flow_pass_strict')]:
    if _mask_col is None:
        _k = _unc_test
    else:
        _k = _unc_test[_unc_test[_mask_col]]
    if len(_k) == 0:
        continue
    _unc_rows.append({
        'stream': 'uncarved',
        'variant': _name,
        'fills': len(_k),
        'retention': len(_k) / len(_unc_test),
        'copyable_pnl': _k['copyable_pnl'].sum(),
        'copyable_roi': _agg_roi(_k, 'copyable_pnl', 'copyable_notional'),
        'actual_pnl': _k['trade_pnl'].sum(),
        'actual_roi': _agg_roi(_k, 'trade_pnl', 'notional'),
        'mean_price': _k['price'].mean(),
        'mean_flow_share': _k['flow_usdc_share_60'].mean(),
    })

_flow_bt = pd.DataFrame(_rows + _unc_rows)
print("Flow filter comparison (test period, multi_filter wallets):")
display(_flow_bt.round(4))

# ── Per-wallet comparison (carved stream) ──────────────────────────────────
_wallet_rows = []
for _w, _g in _ff_base.groupby('wallet'):
    _base_roi = _agg_roi(_g, 'copyable_pnl', 'copyable_notional')
    for _name, _mask_col in [('nonstrict', 'flow_pass_nonstrict'), ('strict', 'flow_pass_strict')]:
        _k = _g[_g[_mask_col]]
        if len(_k) < 5 or _k['copyable_notional'].sum() <= 0:
            continue
        _wallet_rows.append({
            'wallet': _w,
            'variant': _name,
            'n_total': len(_g),
            'n_kept': len(_k),
            'base_roi': _base_roi,
            'filt_roi': _agg_roi(_k, 'copyable_pnl', 'copyable_notional'),
            'delta': _agg_roi(_k, 'copyable_pnl', 'copyable_notional') - _base_roi,
        })
_flow_wallet = pd.DataFrame(_wallet_rows)
if not _flow_wallet.empty:
    for _v in ['nonstrict', 'strict']:
        _sub = _flow_wallet[_flow_wallet['variant'] == _v]
        _improved = (_sub['delta'] > 0).sum()
        print(f"\nPer-wallet carved ({_v}): improved {_improved} of {len(_sub)} wallets")
        display(_sub.sort_values('delta').round(4))

# ── Monthly breakdown (carved) ──────────────────────────────────────────────
_ff_base['month'] = _ff_base['dt'].dt.strftime('%Y-%m')
_month_rows = []
for _mo, _g in _ff_base.groupby('month'):
    for _name, _mask_col in [('baseline', None), ('nonstrict', 'flow_pass_nonstrict'), ('strict', 'flow_pass_strict')]:
        if _mask_col is None:
            _k = _g
        else:
            _k = _g[_g[_mask_col]]
        if len(_k) == 0:
            continue
        _month_rows.append({
            'month': _mo,
            'variant': _name,
            'fills': len(_k),
            'copyable_roi': _agg_roi(_k, 'copyable_pnl', 'copyable_notional'),
            'copyable_pnl': _k['copyable_pnl'].sum(),
        })
_flow_monthly = pd.DataFrame(_month_rows)
if not _flow_monthly.empty:
    print("\nMonthly breakdown (carved):")
    display(_flow_monthly.round(4))

Uncarved fills with PnL: 107,066 / 11,050,037
Flow filter comparison (test period, multi_filter wallets):


,stream,variant,fills,retention,copyable_pnl,copyable_roi,actual_pnl,actual_roi,mean_price,mean_flow_share
0,carved,baseline,3776,1.00,8041.26,0.01,51082.92,0.05,NaN,NaN
1,carved,nonstrict,1034,0.27,4445.11,0.06,-2918.20,-0.02,NaN,NaN
2,carved,strict,307,0.08,1801.83,0.06,7884.52,0.19,NaN,NaN
3,uncarved,baseline,4755562,1.00,12440.76,0.00,394465.50,0.07,0.54,1.16
4,uncarved,nonstrict,859324,0.18,44027.27,0.06,102586.74,0.10,0.35,-1.29
5,uncarved,strict,291848,0.06,42201.69,0.16,68209.91,0.19,0.35,-1.32



Per-wallet carved (nonstrict): improved 6 of 10 wallets


,wallet,variant,n_total,n_kept,base_roi,filt_roi,delta
11,0xc2494b32d65dea3a3a80712589d10bc5733f8a61,nonstrict,358,123,-0.30,-0.64,-0.34
7,0x984ffef12a23e7af5e2cef8e3283e0ab6a6e9a43,nonstrict,314,26,0.06,-0.05,-0.10
17,0xf49614e63fb15383d4a9b717a1be03ad2410fe79,nonstrict,234,19,0.05,-0.04,-0.09
3,0x58194e6e2a57f3f30ad0ac379129c20c03832b4f,nonstrict,292,75,0.01,-0.07,-0.08
13,0xc34b4901be324718c176100e350211d97b199296,nonstrict,304,83,-0.03,-0.03,0.00
9,0x98689f5983898c1da0ec7a9f0864d478f3bee82b,nonstrict,902,298,0.06,0.07,0.01
5,0x8f8883ecf250d19cef876e6be32188b27f2bca10,nonstrict,732,340,-0.07,0.03,0.10
15,0xc3973025163de81e367ad29c21e296e51684bb3b,nonstrict,111,10,0.03,0.35,0.32
1,0x4d1452faf4de78b990e6ec7b9d69217c1df636fd,nonstrict,91,48,0.89,1.24,0.35
0,0x149a9ec31ae097faa33b2f2ae8df659889c149d0,nonstrict,393,12,0.05,1.88,1.83



Per-wallet carved (strict): improved 5 of 9 wallets


,wallet,variant,n_total,n_kept,base_roi,filt_roi,delta
6,0x8f8883ecf250d19cef876e6be32188b27f2bca10,strict,732,68,-0.07,-0.92,-0.85
12,0xc2494b32d65dea3a3a80712589d10bc5733f8a61,strict,358,16,-0.30,-0.99,-0.69
10,0x98689f5983898c1da0ec7a9f0864d478f3bee82b,strict,902,154,0.06,-0.17,-0.23
8,0x984ffef12a23e7af5e2cef8e3283e0ab6a6e9a43,strict,314,9,0.06,-0.09,-0.15
18,0xf49614e63fb15383d4a9b717a1be03ad2410fe79,strict,234,7,0.05,0.18,0.13
14,0xc34b4901be324718c176100e350211d97b199296,strict,304,18,-0.03,0.19,0.22
4,0x58194e6e2a57f3f30ad0ac379129c20c03832b4f,strict,292,16,0.01,0.23,0.22
2,0x4d1452faf4de78b990e6ec7b9d69217c1df636fd,strict,91,13,0.89,1.25,0.37
16,0xc3973025163de81e367ad29c21e296e51684bb3b,strict,111,5,0.03,0.44,0.41



Monthly breakdown (carved):


,month,variant,fills,copyable_roi,copyable_pnl
0,2026-08,baseline,12,-0.67,-1618.72
1,2026-09,baseline,2437,0.05,13493.38
2,2026-09,nonstrict,760,0.06,3341.78
3,2026-09,strict,235,-0.05,-1063.18
4,2026-10,baseline,1327,-0.01,-3833.40
5,2026-10,nonstrict,274,0.04,1103.33
6,2026-10,strict,72,0.34,2865.02


In [57]:
wallet_cohorts['predicting_group'].sort_values('total_pnl', ascending=False, key=abs)

,wallet,num_buckets,trade_count,total_notional,total_pnl,copyable_pnl,num_markets,median_dt,median_roi,average_roi,max_drawdown,max_copyable_drawdown,buy_pnl,buy_quantity,sell_pnl,sell_quantity,buy_notional,sell_notional,buy_copyable_quantity,buy_copyable_pnl,sell_copyable_quantity,sell_copyable_pnl,buy_roi,sell_roi,buy_copyable_notional,top5_pnl_pct,top10_pnl_pct,worst5_pnl_pct,positive_bucket_share,top_market_pnl_pct,top_market_abs_pnl_pct,market_pnl_hhi,pnl_volatility,estimated_sharpe,estimated_buy_sharpe,estimated_copyable_buy_sharpe,copyable_pnl_similarity,copyable_pnl_similarity_profit,copyable_pnl_similarity_loss,copyable_pnl_similarity_20m_100,copyable_pnl_similarity_profit_20m_100,copyable_pnl_similarity_loss_20m_100,max_drawdown_to_pnl,max_copyable_drawdown_to_copyable_pnl,return,max_drawdown_all,total_pnl_all,max_drawdown_all_to_pnl_all,copyable_pnl_factor,copyable_roi,buy_copyable_roi,buy_copyable_factor
11,0xc2494b32d65dea3a3a80712589d10bc5733f8a61,1986,4387,684481.40,44850.68,2629.60,1026,2026-05-22 01:10:00+00:00,0.22,0.14,15557.31,11052.92,44850.68,1410218.49,0.00,0.00,684481.40,0.00,287611.47,2629.60,0.00,0.00,0.07,0.00,139598.73,0.33,0.57,-0.37,0.55,0.08,0.01,0.18,4.82,2.43,2.43,0.31,0.60,0.58,0.61,NaN,NaN,NaN,0.35,4.20,0.07,13887.45,44826.63,0.31,0.06,0.01,0.02,0.06
8,0xc34b4901be324718c176100e350211d97b199296,292,409,237277.76,36344.51,10494.50,256,2026-08-11 01:50:00+00:00,0.37,0.16,8042.18,4802.48,36344.51,498592.67,0.00,0.00,237277.76,0.00,165538.86,10494.50,0.00,0.00,0.15,0.00,78779.12,0.40,0.73,-0.32,0.62,0.10,0.02,0.16,7.55,2.87,2.87,1.47,0.65,0.62,0.68,NaN,NaN,NaN,0.22,0.46,0.15,8069.54,36086.93,0.22,0.29,0.05,0.13,0.29
9,0xf49614e63fb15383d4a9b717a1be03ad2410fe79,55,216,109442.52,27422.59,5023.80,44,2026-08-02 10:35:00+00:00,0.02,0.12,5458.99,5458.99,27422.59,144615.57,0.00,0.00,109442.52,0.00,44736.30,5023.80,0.00,0.00,0.25,0.00,33855.65,0.96,1.17,-0.22,0.89,0.41,0.29,0.27,19.30,4.64,4.64,1.74,0.57,0.49,1.00,NaN,NaN,NaN,0.20,1.09,0.25,6815.18,23865.28,0.29,0.18,0.02,0.15,0.18
1,0x73492e9ea5e7423cc0bb10a7169a0c81972dcc3a,111,828,216833.67,23563.52,20204.95,70,2026-07-04 19:00:00+00:00,0.16,0.19,13141.82,8141.01,23563.52,337208.08,0.00,0.00,216833.67,0.00,219991.36,20204.95,0.00,0.00,0.11,0.00,141460.23,1.36,1.84,-0.94,0.78,0.40,0.08,0.50,16.81,2.61,2.61,2.92,0.90,0.92,0.88,NaN,NaN,NaN,0.56,0.40,0.11,10035.29,23797.11,0.42,0.86,0.16,0.14,0.86
7,0x8f8883ecf250d19cef876e6be32188b27f2bca10,37,108,26679.91,17882.73,1813.16,33,2026-08-16 14:15:00+00:00,-1.00,0.47,4254.52,4135.94,17882.73,90923.45,0.00,0.00,26679.91,0.00,38012.55,1813.16,0.00,0.00,0.67,0.00,11154.12,1.36,1.51,-0.30,0.49,0.52,0.24,0.49,29.89,3.39,3.39,1.00,0.63,0.63,0.84,NaN,NaN,NaN,0.24,2.28,0.67,5101.89,15929.41,0.32,0.10,0.05,0.16,0.10
4,0x2a8a536cb3448f6e0c94d4a3ed7ede7855c1cb94,68,384,69682.25,15699.40,7409.64,47,2026-07-05 20:55:00+00:00,0.25,0.22,3276.44,2759.60,15699.40,105178.45,0.00,0.00,69682.25,0.00,43279.67,7409.64,0.00,0.00,0.23,0.00,28673.41,0.82,1.07,-0.38,0.81,0.47,0.24,0.29,10.71,4.43,4.43,3.42,0.89,0.93,0.59,NaN,NaN,NaN,0.21,0.37,0.23,3737.97,8910.98,0.42,0.47,0.10,0.26,0.47
0,0xc3973025163de81e367ad29c21e296e51684bb3b,29,40,45884.03,11304.85,7367.98,22,2026-08-26 08:35:00+00:00,0.33,0.32,4000.00,4000.00,11304.85,67219.21,0.00,0.00,45884.03,0.00,55055.43,7367.98,0.00,0.00,0.25,0.00,37581.00,1.05,1.38,-0.51,0.87,0.44,0.22,0.32,16.39,2.66,2.66,2.12,0.96,0.93,NaN,NaN,NaN,NaN,0.35,0.54,0.25,4000.00,11204.25,0.36,0.65,0.21,0.20,0.65
2,0x149a9ec31ae097faa33b2f2ae8df659889c149d0,59,78,27622.11,6461.81,4913.92,56,2026-08-05 12:00:00+00:00,0.43,0.21,1408.58,1659.52,6461.81,42014.38,0.00,0.00,27622.11,0.00,35520.63,4913.92,0.00,0.00,0.23,0.00,23352.83,0.51,0.88,-0.60,0.82,0.14,0.07,0.12,6.19,3.77,3.77,3.19,0.97,0.95,1.00,NaN,NaN,NaN,0.22,0.34,0.23,1408.58,6457.05,0.22,0.76,0.16,0.21,0.76
6,0x4d1452faf4de78b990e6ec7b9d69217c1df636fd,19,55,10337.64,4538.79,1137.26,7,2026-08-18 23:55:00+00:00,0.42,0.31,576.27,351.63,4538.79,16295.74,0

In [58]:
_ff_base.groupby('wallet')['copyable_pnl'].sum().sort_values(key=abs)[-5:].reset_index()['wallet'].to_list()

['0x4d1452faf4de78b990e6ec7b9d69217c1df636fd',
 '0xc3973025163de81e367ad29c21e296e51684bb3b',
 '0x8f8883ecf250d19cef876e6be32188b27f2bca10',
 '0xc2494b32d65dea3a3a80712589d10bc5733f8a61',
 '0x149a9ec31ae097faa33b2f2ae8df659889c149d0']

In [59]:
# ── Cumulative copyable PnL by flow filter variant ─────────────────────────
import plotly.graph_objects as go

_ff_base['bucket'] = _ff_base['dt'].dt.floor('5min')
_ff_base = _ff_base.sort_values('bucket')

fig = go.Figure()

_variant_masks = {
    'baseline': np.ones(len(_ff_base), dtype=bool),
    'nonstrict': _ff_base['flow_pass_nonstrict'].values,
    'strict': _ff_base['flow_pass_strict'].values,
}
_colors = {'baseline': '#636efa', 'nonstrict': '#2ca02c', 'strict': '#d62728'}
_dashes = {'baseline': 'solid', 'nonstrict': 'dash', 'strict': 'dot'}

for w in _ff_base.groupby('wallet')['copyable_pnl'].sum().sort_values(key=abs)[-5:].reset_index()['wallet'].to_list():
    for _name, _mask in _variant_masks.items():
        _sub = _ff_base[
            _mask
            & (_ff_base['wallet'] == w)
            ].copy()
        if _sub.empty:
            continue
        _sub['cum_pnl'] = _sub['copyable_pnl'].cumsum()
        fig.add_trace(go.Scatter(
            x=_sub['bucket'],
            y=_sub['cum_pnl'],
            mode='lines',
            name=f"{w}: {_name} (n={_mask.sum():,})",
            line=dict(color=_colors[_name], dash=_dashes[_name], width=2),
        ))

fig.update_layout(
    title='Cumulative copyable PnL — flow filter variants (test period, flow_filter wallets)',
    xaxis_title='Time',
    yaxis_title='Cumulative copyable PnL (USDC)',
    hovermode='x unified',
    template='plotly_white',
)
fig.show()